## Parcel Update - Spring 2021
* Amy Fish, afish@trpa.org
* Emily Ulrich, eulrich@trpa.org
* Mason Bindl, mbindl@trpa.org

### Get Parcel Data

* El Dorado County Data:
    * https://see-eldorado.edcgov.us/ugotnetextracts/
    * http://gem.edcgov.us/arcgis/rest/services/extracts/geoservices8SQL/MapServer/1
    * Contacts: Jose Crummett <jose.crummett@edcgov.us> and Mark La Loggia <mark.laloggia@edcgov.us>  

* Placer County Data:
     * email: RGoodner@placer.ca.gov for the table with other attributes
     * http://gis-placercounty.opendata.arcgis.com/datasets/e49d7e972674481cbb5ecc047b8f98eb_0
     * https://services6.arcgis.com/PArfeTGcwA9RGNzN/arcgis/rest/services/Parcels/FeatureServer/0
     * Contacts: Cody Monroe <cmonroe@placer.ca.gov> and Ryan Goodner-Belli <RGoodner@placer.ca.gov> and Dennis Decelle       <DDeCelle@placer.ca.gov>

* Washoe County Data:
     * http://explore-washoe.opendata.arcgis.com/
     * https://gis.washoecounty.us/arcgis/rest/services/OpenData/OpenData/FeatureServer/0
     * https://services.arcgis.com/iCGWaR7ZHc5saRIl/arcgis/rest/services/WashoeCountyParcel_LandUseCodes_Table/FeatureServer/0
     * Contact: Dixie Rudebusch <DRudebusch@washoecounty.us>

* Carson City County Data:
     * http://data-carsoncity.opendata.arcgis.com/datasets/parcels
     * https://gis.carson.org/arcgis/rest/services/CarsonCity/CarsonCityNV_OpenData/FeatureServer/36
     * http://www.ccapps.org/profoundui/start?pgm=aspgm/asr840cl
     * Search records FROM APN: 007-011-01 TO APN: 007-031-17 
     * Contact: Matthew Lawton <mlawton@carson.org>

* Douglas County Data:
     * email: GIS@douglasnv.us
     * Contacts: Leah Montoya and Matt Richardson <MRichardson@douglasnv.us> phone:775-782-9894

### Import Modules, Set Local Variables, and Define Functions

In [2]:
import arcpy, sys, datetime, os, traceback
# Set Data Source GDB Names
county_gdb = "Original.gdb"
TRPA_staging_gdb = "Staging.gdb"
w_folder = '2021_04'  # Change based on new file folder name
w_folder_old = '2020_10' # Change based on old file folder name
w_original = '\\Original\\'
w_modified = '\\Modified\\'
n_parcels = 'Parcels'
sep = '-'
County_List = ['Carson', 'Douglas', 'El_Dorado', 'Placer', 'Washoe']
County_ABR_List = ['CC', 'DG', 'EL', 'PL', 'WA']
state_loc = ['CA', 'NV']
python_version = "PYTHON3"

# set base feature classes
ParcelLayer = "F:\\GIS\\ParcelUpdate\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
ParcelPoint = "F:\\GIS\\ParcelUpdate\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Point"

# Placer extent feature class
PlacerExtent = "F:\\GIS\\ParcelUpdate\\2019_11\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_TRCD"
ParcelsTRCD = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_TRCD"

# File paths
sdeBase = "F:\\GIS\\DB_CONNECT\\Vector.sde"
sdeTabular = "F:\\GIS\\DB_CONNECT\\Tabular.sde"
#sde feature classes
sde_FireDistrict = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_NRCSSoils1974 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_HydroArea = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed = sdeBase + "\\sde.SDE.Water\\sde.SDE.Priority"
sde_RegionalLandUse = sdeBase + "\\sde.SDE.Planning\\sde.SDE.RegionalLandUse"
sde_LocalPlan = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_Zoning =  sdeBase + "\\sde.SDE.Planning\\sde.SDE.Zoning_LocalPlan"
sde_SpecialDistrict = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_TownCenter = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987 = sdeBase + "\\sde.SDE.Planning\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_UrbanArea = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_CSLT = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"

# in memory files
wk_memory = "memory" + "\\"
ParcelPoint_FireDistrict = wk_memory + "\\ParcelPoint_FireDistrict"
ParcelPoint_Soils74 = wk_memory + "\\ParcelPoint_Soils74"
ParcelPoint_Soils03 = wk_memory + "\\ParcelPoint_Soils03"
ParcelPoint_HydroArea = wk_memory + "\\ParcelPoint_HydroArea"
ParcelPoint_Watershed = wk_memory + "\\ParcelPoint_Watershed"
ParcelPoint_RegionalLandUse = wk_memory + "\\ParcelPoint_RegionalLandUse"
ParcelPoint_LocalPlan = wk_memory + "\\ParcelPoint_LocalPlan"
ParcelPoint_TownCenter = wk_memory + "\\ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer = wk_memory + "\\ParcelPoint_TownCenterBuffer"
ParcelPoint_Zoning = wk_memory + "\\ParcelPoint_Zoning"
ParcelPoint_SpecialDistrict = wk_memory + "\\ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987 = wk_memory + "\\ParcelPoint_Index1987"
ParcelPoint_PstlTown = wk_memory + "\\ParcelPoint_PstlTown"
ParcelPoint_PstlZip = wk_memory + "\\ParcelPoint_PstlZip"
ParcelPoint_CSLT = wk_memory + "\\ParcelPoint_CSLT"

## Get Data

### Washoe County - Map Service to Feature Class

#### Pseudo Code
* Please test before using
* Move to step 3

In [ ]:
import arcpy
# overwrite outputs
arcpy.env.overwriteOutput = True

# setup bse URL of feature service
baseURL = "https://services.arcgis.com/iCGWaR7ZHc5saRIl/arcgis/rest/services/WashoeCountyParcel_LandUseCodes_Table/FeatureServer/0"
# specify all features
where = "1=1"
# specify fields
fields = "CITY,ACRES,COUNTY"
# query the feature service for all features with just the fields we want and project to NAD83 UTM10N on the fly
query = "?where={}&outFields={}&returnGeometry=true&f=json&outSR=26910".format(where, fields)
# setup output feature class
outfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\WA\\Download\\WA_Original.shp"
# JSON feature set URL
fsURL = baseURL + query

# create feature set from queried feature service returned as json
fs = arcpy.FeatureSet()
fs.load(fsURL)

# export feature set to feature class
arcpy.CopyFeatures_management(fs, outfc)

### Get Data From Map Service with Record Limit

#### Pseudo Code
* Please test before using
* Move to step 3

In [ ]:
import arcpy
import urllib2
import json


# Setup
arcpy.env.overwriteOutput = True
baseURL = "https://services.arcgis.com/iCGWaR7ZHc5saRIl/arcgis/rest/services/WashoeCountyParcel_LandUseCodes_Table/FeatureServer/0"
fields = "*"
outdata = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\WA\\Download\\DataDownload.gdb\\WA_Download"

# Get record extract limit
urlstring = baseURL + "?f=json"
j = urllib2.urlopen(urlstring)
js = json.load(j)
maxrc = int(js["maxRecordCount"])
print "Record extract limit: %s" % maxrc

# Get object ids of features
where = "1=1"
urlstring = baseURL + "/query?where={}&returnIdsOnly=true&f=json&outSR=26910".format(where)
j = urllib2.urlopen(urlstring)
js = json.load(j)
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print "Number of target records: %s" % numrec

# Gather features
print "Gathering records..."
fs = dict()
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
    torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print "  {}".format(where)
    urlstring = baseURL + "/query?where={}&returnGeometry=true&outFields={}&f=json".format(where,fields)
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print "Saving features..."
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print "Done!"

## Translate County Data

### Select County to be proccessed to set field list

In [2]:
while True:
    try:
        UserCountyResponse = input("Enter a County Name: ")
        if UserCountyResponse.lower() == "carson":
            UserCountyResponse = (County_List[0])
            County_ABR = (County_ABR_List[0])
            fld_list = ['APN', 'Legal_Owner', 'Mailing_Address_Line_1', 'Mailing_Address_Line_2',
                        'Land_Value', 'Improvements_Value', 'Location_Number', 'Location_Unit___s_', 
                        'Location_Direction', 'Location_or_Street_Name', 'UPDATED', 'Land_Use_Code']
            break
        elif UserCountyResponse.lower() == "douglas":
            UserCountyResponse = (County_List[1])
            County_ABR = (County_ABR_List[1])
            fld_list = ['APN', 'PIN', 'PANAME', 'PMADD2', 'PMADD1', 'PMCTST', 'PZIP', 'YLANDV',
                        'YIMPRV', 'PLOC_', 'PLOCU_', 'PLOCDR', 'PLOCNM', 'PUPDDT', 'YLDUSE', 
                        'PLOCTP', 'PAPPYR', 'PTOWN', 'P_DWEL', 'PBEDS', 'PBATHS']
            break
        elif UserCountyResponse.lower() == "el_dorado" or UserCountyResponse.lower() == "el dorado":
            UserCountyResponse = (County_List[2])
            County_ABR = (County_ABR_List[2])
            fld_list = ['PRCL_ID', 'OWNER_NAME', 'MAIL_ADDR1', 
                        'MAIL_ADDR2', 'MAIL_ADDR3', 'MAIL_ADDR4',
                        'LAND_VAL', 'STRUCT_VAL', 'ADDRSTNBR', 'ADDRUNITNB', 
                        'ADDRSTNAME', 'ADDRSTTYPE', 'POLY_CREAT',
                        'USECD_1']
            break
        elif UserCountyResponse.lower() == "placer":
            UserCountyResponse = (County_List[3])
            County_ABR = (County_ABR_List[3])
            fld_list = ['APN', 'GISAPN', 'OWNER1', 'OWNER2', 'ADR2', 
                        'CITY', 'STATE', 'ZIP', 'LANDVALUE', 'STRUCTURE',
                        'STREETNUM', 'SP_APT', 'STREETDIR', 'STREETNAME', 
                        'STREETTYPE', 'USE_CD', 'COMMUNITY', 'TRANSACTIO']
            break
        elif UserCountyResponse.lower() == "washoe":
            UserCountyResponse = (County_List[4])
            County_ABR = (County_ABR_List[4])
            fld_list = ['PIN', 'APN', 'FIRSTNAME', 'LASTNAME', 'MAILING1', 
                        'MAILING2', 'MAILCITY', 'MAILSTATE',
                        'MAILZIP', 'LANDASS', 'BUILDASS',
                        'STREETNUM', 'STREETDIR', 'STREET', 'LAND_USE', 
                        'TAXYEAR', 'CITY', 'SITUSZIP']
            break
        else:
            print ("County entered is not valid. Try a county listed below")
            for x in County_List:
                print (x)
            print('\n')
            continue
    except Exception as e:
        print (e)
        sys.exit()
print ("{} County will be processed".format(UserCountyResponse))

Enter a County Name: El Dorado
El_Dorado County will be processed


#### Set Workspace Enviroment

In [3]:
base_working_folder = os.path.join(r'F:\GIS\ParcelUpdate', w_folder, UserCountyResponse)
arcpy.env.workspace = base_working_folder + w_original + county_gdb
print (arcpy.env.workspace)

F:\GIS\ParcelUpdate\2021_04\El_Dorado\Original\Original.gdb


#### Check for ArcInfo license

In [4]:
try:
    license_info = arcpy.ProductInfo()
    if not license_info == "NotInitialized":
        print ("Product License: {}".format(arcpy.ProductInfo()))
    else:
        raise Exception
except Exception:
    print ("ArcGIS Product License Error: {}".format(arcpy.ProductInfo()))
    sys.exit()

Product License: ArcInfo


#### Set Default Extent

In [5]:
arcpy.env.extent = sde_TRPAboundary
print ("The feature extent is being used in env {}".format(str(arcpy.env.extent)))
arcpy.env.spatialGrid1 = s_grid_1 = 5400
arcpy.env.spatialGrid2 = s_grid_2 = 16200
arcpy.env.spatialGrid3 = s_grid_3 = 48600

The feature extent is being used in env 737668.911313948 4288193.03147232 770952.921734459 4357302.88519489 NaN NaN NaN NaN


In [5]:
# ****IMPORTANT**** Use for Placer County only
arcpy.env.extent = PlacerExtent
print ("The feature extent is being used in env {}".format(str(arcpy.env.extent)))
arcpy.env.spatialGrid1 = s_grid_1 = 5400
arcpy.env.spatialGrid2 = s_grid_2 = 16200
arcpy.env.spatialGrid3 = s_grid_3 = 48600

The feature extent is being used in env 731872.520068198 4290435.04320323 772577.109367072 4358086.4098659 NaN NaN NaN NaN


#### Set Default Coordinate System

In [6]:
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 UTM Zone 10N")
sr = arcpy.env.outputCoordinateSystem
print ("Spatial Reference: {}".format(sr.name))

Spatial Reference: NAD_1983_UTM_Zone_10N


#### Remove M and Z values

In [7]:
arcpy.env.outputZFlag = "Disabled"
arcpy.env.outputMFlag = "Disabled"
arcpy.env.overwriteOutput = True

#### Set Local Variables

In [8]:
# Set local variables
out_gdb_name = County_ABR + "_" + TRPA_staging_gdb 
modified_folder = base_working_folder + w_modified
original_folder = base_working_folder + w_original
modified_fgd = modified_folder + out_gdb_name
original_fgd = original_folder + out_gdb_name
# County local variables
parcel_in_data = UserCountyResponse + '_' + n_parcels
fc_outname = parcel_in_data
fdata_name = n_parcels
fdataset = os.path.join(modified_fgd, n_parcels)
parcel_out_data = fdataset + '\\' + fc_outname
print ("Features saved here: {}".format(parcel_out_data))

Features saved here: F:\GIS\ParcelUpdate\2021_04\El_Dorado\Modified\EL_Staging.gdb\Parcels\El_Dorado_Parcels


#### Track Progress

In [9]:
# Start timer for process
start_time = datetime.datetime.today()
print ("Script started: {}".format(start_time))
# Create  and open log file.
complete_txt_path = os.path.join(base_working_folder, fc_outname + ".txt")
print (complete_txt_path)
log = open(complete_txt_path, "w")
# Write results to txt file
log.write("Log: " + str(start_time) + "\n")
log.write("\n")
log.write("Begin process:\n")
log.write("Process started at: " + str(start_time) + "\n")
log.write("\n")

Script started: 2021-04-23 12:17:01.392153
F:\GIS\ParcelUpdate\2021_04\El_Dorado\El_Dorado_Parcels.txt


1

#### Project Placer Dataset

In [10]:
# input data is in NAD 1983 UTM Zone 11N coordinate system
input_features = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Placer\\Original\\Temp.gdb\\Placer_Parcels_UnProjected"

# output data
output_feature_class = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Placer\\Original\\Original.gdb\\Placer_Parcels"

# create a spatial reference object for the output coordinate system
out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N')

# run the tool
arcpy.Project_management(input_features, output_feature_class, out_coordinate_system)

<Result 'F:\\GIS\\ParcelUpdate\\2021_04\\Placer\\Original\\Original.gdb\\Placer_Parcels'>

#### Describe original feature class and its spatial reference

In [10]:
desc = arcpy.Describe(parcel_in_data)
spatialref = desc.spatialReference
print ("Original Spatial Reference: {}".format(spatialref.name))
log.write("Original Spatial Reference: {}".format(spatialref.name) + "\n")

Original Spatial Reference: NAD_1983_StatePlane_California_II_FIPS_0402_Feet


77

#### Create File GDB and Feature Class for Chosen County

In [11]:
while True:
    try:
        CreateGDBResponse = input("Create new GDB and Feature Class? (Yes/No)")
        if CreateGDBResponse.lower() == "yes":
            # Check if FileGDB exists. If not Execute CreateFileGDB
            if not arcpy.Exists(modified_fgd):
                arcpy.CreateFileGDB_management(modified_folder, out_gdb_name)
                print ("File Geodatabase {} created!".format(out_gdb_name))
                log.write("File Geodatabase {} created!".format(out_gdb_name) + "\n")
            # Execute CreateFileGDB will not be run
            else:
                print ("File Geodatabase {} already exists!".format(out_gdb_name))
                log.write("File Geodatabase {} already exists!".format(out_gdb_name) + "\n")
                
            # Check if Feature Dataset exists. Execute Copy
            if not arcpy.Exists(fdataset):
                arcpy.CreateFeatureDataset_management(modified_fgd, fdata_name, sr)
                print ("Feature Dataset Created!")
            # Execute Delete Management then execute CopyFeatures Management
            else:
                print ("Feature Dataset {} already exists!".format(fdata_name))
                arcpy.Delete_management(fdataset)
                print ("Deleted {}!".format(fdata_name))
                log.write("Deleted {}".format(fdata_name) + "\n")
                arcpy.CreateFeatureDataset_management(modified_fgd, fdata_name, sr)
                print ("Created Dataset {}".format(fdata_name))
            fcs = arcpy.ListFeatureClasses()
            for fc in fcs:
                # Copy features from the workspace to FGD
                arcpy.CopyFeatures_management(fc, parcel_out_data)
                print ("Added {}".format(fc))
                log.write("Added {}".format(fc) + "\n")
                # Execute GP tool to get count of records in FeatureClass
                result = arcpy.GetCount_management(parcel_out_data)
                oldresult = arcpy.GetCount_management(parcel_in_data)
                oldcount = int(oldresult.getOutput(0))
                count = int(result.getOutput(0))
                print ("Number of records copied {0} out of the {1}".format(count, oldcount))
                log.write("Number of records copied {0} out of the {1}".format(count, oldcount) + "\n")
                if count > 0:
                    # Create list of field names in FC that is being processed
                    Fc_fields = arcpy.ListFields(parcel_out_data)
                    for field in Fc_fields:
                        print ("{0} type: {1} length: {2}".format(field.name, field.type, field.length))
                        log.write("Name: {}".format(field.name) + ", ")
                        log.write("Type: {}".format(field.type) + ", ")
                        log.write("Length: {}".format(field.length) + "\n")
                    # Add all fields
                    fields = [
                        # Fields for APN & PPNO
                        ("APN_NEW", "TEXT", "", "", "16", "APN", "NULLABLE", "NON_REQUIRED", ""),
                        ("PPNO_NEW", "DOUBLE", "", "0", "", "PPNO", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for local address table
                        ("HSE_NUMBR_NEW", "Text", "", "", "25", "Lot #", "NULLABLE", "NON_REQUIRED", ""),
                        ("UNIT_NUMBR_NEW", "Text", "", "", "12", "Unit #", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_DIR_NEW", "Text", "", "", "2", "St Direction", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_NAME_NEW", "Text", "", "", "100", "St Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_SUFFIX_NEW", "Text", "", "", "6", "St Suffix", "NULLABLE", "NON_REQUIRED", ""),
                        ("APO_ADDRESS_NEW", "Text", "", "", "100", "Full Address", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_TOWN_NEW", "TEXT", "", "", "25", "Town", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_STATE_NEW", "TEXT", "", "", "2", "State", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_ZIP5_NEW", "TEXT", "", "", "5", "ZIP5", "NULLABLE", "NON_REQUIRED", ""),
                        # fields for mailing address
                        ("OWN_FIRST_NEW", "TEXT", "", "", "50", "Owner First Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWN_LAST_NEW", "TEXT", "", "", "100", "Owner Last Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWN_FULL_NEW", "TEXT", "", "", "100", "Owner Full Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ADD1_NEW", "TEXT", "", "", "100", "Mail Address 1", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ADD2_NEW", "TEXT", "", "", "100", "Mail Address 2", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_CITY_NEW", "TEXT", "", "", "50", "Mail City", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_STATE_NEW", "TEXT", "", "", "2", "Mail State", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ZIP5_NEW", "TEXT", "", "", "5", "Mail ZIP5", "NULLABLE", "NON_REQUIRED", ""),
                        ("JURISDICTION_NEW", "TEXT", "", "", "4", "Jurisdiction", "NULLABLE", "NON_REQUIRED", ""),
                        ("COUNTY", "TEXT", "", "", "2", "County", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWNERSHIP_TYPE", "TEXT", "", "", "12", "Ownership  Type", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for land use, soil, watershed, etc...
                        ("COUNTY_LANDUSE_CODE", "TEXT", "", "", "4", "County Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("COUNTY_LANDUSE_DESCRIPTION", "TEXT", "", "", "150", "County Land Use Description", "NULLABLE", "NON_REQUIRED", ""),
                        ("TRPA_LANDUSE_DESCRIPTION", "TEXT", "", "", "50", "TRPA Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("REGIONAL_LANDUSE", "TEXT", "", "", "50", "TRPA Regional Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("UNITS_NEW", "TEXT", "", "", "5", "Units", "NULLABLE", "NON_REQUIRED", ""),
                        ("YEAR_BUILT_NEW", "TEXT", "", "", "5", "Year Built", "NULLABLE", "NON_REQUIRED", ""),
                        ("BEDROOMS_NEW", "TEXT", "", "", "5", "Bedrooms", "NULLABLE", "NON_REQUIRED", ""),
                        ("BATHROOMS_NEW", "TEXT", "", "", "5", "Bathrooms", "NULLABLE", "NON_REQUIRED", ""),
                        ("BUILDING_SQFT_NEW", "DOUBLE", "12", "2", "", "Building (sq.ft.)", "NULLABLE", "NON_REQUIRED", ""),
                        ("ALLOWABLE_COVERAGE_BAILEY_SQFT", "DOUBLE", "12", "2", "", "Allowable Coverage (Bailey, sq.ft.)", "NULLABLE", "NON_REQUIRED", ""),
                        ("IMPERVIOUS_SURFACE_SQFT", "DOUBLE", "12", "2", "", "Impervious Surface (LiDAR, sq.ft.)", "NULLABLE", "NON_REQUIRED", ""),
                        ("SOIL_1974", "TEXT", "", "", "5", "Soils 1974", "NULLABLE", "NON_REQUIRED", ""),
                        ("SOIL_2003", "TEXT", "", "", "5", "Soils 2003", "NULLABLE", "NON_REQUIRED", ""),
                        ("HRA_NAME", "TEXT", "", "", "30", "Hydrologic Resource Area", "NULLABLE", "NON_REQUIRED", ""),
                        ("WATERSHED_NUMBER", "SHORT", "", "", "4", "Watershed #", "NULLABLE", "NON_REQUIRED", ""),
                        ("WATERSHED_NAME", "TEXT", "", "", "30", "Watershed Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("PRIORITY_WATERSHED", "TEXT", "", "", "2", "Priority Watershed", "NULLABLE", "NON_REQUIRED", ""),
                        ("FIREPD", "TEXT", "", "", "25", "Fire Protection District", "NULLABLE", "NON_REQUIRED", ""),
                        ("WITHIN_TRPA_BNDY", "SHORT", "", "", "1", "Within TRPA Boundary", "NULLABLE", "NON_REQUIRED", ""),
                        ("LITTORAL", "SHORT", "", "", "1", "Littoral", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Parcel Value table
                        ("AS_LANDVALUE_NEW", "LONG", "9", "", "", "Assessed Land", "NULLABLE", "NON_REQUIRED", ""),
                        ("AS_IMPROVALUE_NEW", "LONG", "9", "", "", "Assessed Improvement", "NULLABLE", "NON_REQUIRED", ""),
                        ("AS_SUM_NEW", "LONG", "9", "", "", "Assessed Sum", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_LANDVALUE_NEW", "LONG", "9", "", "", "Tax Land", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_IMPROVALUE_NEW", "LONG", "9", "", "", "Tax Improvement", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_SUM_NEW", "LONG", "9", "", "", "Tax Sum", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_YEAR_NEW", "TEXT", "", "", "4", "Tax Year", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Planning purposes
                        ("PLAN_ID", "TEXT", "", "", "8", "Plan ID", "NULLABLE", "NON_REQUIRED", ""),
                        ("PLAN_NAME", "TEXT", "", "", "40", "Plan Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("ZONING_ID", "TEXT", "", "", "50", "Zoning ID", "NULLABLE", "NON_REQUIRED", ""),
                        ("ZONING_DESCRIPTION", "TEXT", "", "", "500", "Zoning Description", "NULLABLE", "NON_REQUIRED", ""),
                        ("TOLERANCE_ID", "TEXT", "", "", "50", "Tolerance ID", "NULLABLE", "NON_REQUIRED", ""),
                        ("TOWN_CENTER", "TEXT", "", "", "50", "Town Center", "NULLABLE", "NON_REQUIRED", ""),
                        ("LOCATION_TO_TOWNCENTER", "TEXT", "", "", "50", "Location Relative to Town Center", "NULLABLE", "NON_REQUIRED", ""),
                        ("INDEX_1987", "TEXT", "", "", "10", "1987 Parcel Map Index", "NULLABLE", "NON_REQUIRED", ""),
                        ("LOCAL_PLAN_HYPERLINK", "TEXT", "", "", "255", "Local Plan Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("DESIGN_GUIDELINES_HYPERLINK", "TEXT", "", "", "255", "Design Guideline Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("LTINFO_HYPERLINK", "TEXT", "", "", "150", "Parcel Details", "NULLABLE", "NON_REQUIRED", ""),
                        ("INDEX_1987_HYPERLINK", "TEXT", "", "", "255", "1987 Parcel Map Index Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAZ", "DOUBLE", "6", "2", "", "Transportation Analysis Zone", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Parcel Size
                        ("PARCEL_ACRES_NEW", "DOUBLE", "6", "2", "", "Acres", "NULLABLE", "NON_REQUIRED", ""),
                        ("PARCEL_SQFT_NEW", "DOUBLE", "12", "2", "", "Square Feet", "NULLABLE", "NON_REQUIRED", ""),
                        # temp field ot check for duplicates
                        ("DUPLICATE", "SHORT", "", "", "1", "", "NULLABLE", "NON_REQUIRED", "")
                    ]
                    # Create the fields using the above parameters
                    for field in fields:
                        arcpy.AddField_management(*(parcel_out_data,) + field)
                        print("{} field created".format(field))
                    break
                else:
                    print("{} No features in feature class. Exit Script".format(count))
                    exit()
            break
        elif CreateGDBResponse.lower() == "no":
            # Check if FileGDB exists.
            if arcpy.Exists(modified_fgd):
                print ("It's a go!")
                break
            else:
                print ("Staging Geodatabase doesn't exist. Needs to be created. Type Yes")
                continue
        else:
            print ("Staging Geodatabase doesn't exist. Needs to be created. Type Yes")
            continue
    except Exception as e:
        print (e)
        exit()

# print "Begin list of all fields that exists"
Strd_fields = arcpy.ListFields(parcel_out_data)
for field in Strd_fields:
    # print ("{0} type: {1} length: {2}".format(field.name, field.type, field.length))
    log.write("Name: {}".format(field.name) + ", ")
    log.write("Type: {}".format(field.type) + ", ")
    log.write("Length: {}".format(field.length) + "\n")

# Get the list of field names in file and list of predefined field names
# that will be calculated. Missing fields will not crash script, only close.
field_names = [field.name for field in arcpy.ListFields(parcel_out_data)]
match_fields = [i for i in fld_list if i in field_names]
unmatch_fields = [i for i in fld_list if i not in field_names]


Create new GDB and Feature Class? (Yes/No)Yes
File Geodatabase EL_Staging.gdb created!
Feature Dataset Created!
Added El_Dorado_Parcels
Number of records copied 30758 out of the 30758
OBJECTID_1 type: OID length: 4
Shape type: Geometry length: 0
OBJECTID type: Integer length: 4
POLY_ID type: Integer length: 4
PRCL_ID type: String length: 50
POLY_CLASS type: Integer length: 4
POLY_CREAT type: Date length: 8
POLY_DESCR type: String length: 50
EXCPTN_LIT type: String length: 15
TAX_CD type: Double length: 8
TAXCDDESCR type: String length: 50
USECD_1 type: String length: 2
USECDCL_1 type: String length: 3
USECDTY_1 type: String length: 3
USECDLIT_1 type: String length: 50
USECD_2 type: String length: 2
USECDCL_2 type: String length: 3
USECDTY_2 type: String length: 3
USECDLIT_2 type: String length: 50
ACREAGE type: Double length: 8
LEGAL_DESC type: String length: 50
SUBDIV_LIT type: String length: 75
NEIGHBORHD type: String length: 4
ADDRSTNBR type: Double length: 8
HOUSE_SFX type: String 

('LTINFO_HYPERLINK', 'TEXT', '', '', '150', 'Parcel Details', 'NULLABLE', 'NON_REQUIRED', '') field created
('INDEX_1987_HYPERLINK', 'TEXT', '', '', '255', '1987 Parcel Map Index Hyperlink', 'NULLABLE', 'NON_REQUIRED', '') field created
('TAZ', 'DOUBLE', '6', '2', '', 'Transportation Analysis Zone', 'NULLABLE', 'NON_REQUIRED', '') field created
('PARCEL_ACRES_NEW', 'DOUBLE', '6', '2', '', 'Acres', 'NULLABLE', 'NON_REQUIRED', '') field created
('PARCEL_SQFT_NEW', 'DOUBLE', '12', '2', '', 'Square Feet', 'NULLABLE', 'NON_REQUIRED', '') field created
('DUPLICATE', 'SHORT', '', '', '1', '', 'NULLABLE', 'NON_REQUIRED', '') field created


#### Get Year

In [12]:
# Get current year
current_year = datetime.datetime.today().year

#### List of variation of all federal, state and local agencies found in assessor datasets

In [13]:
# ****IMPORTANT**** Use this script to generate the federal, state and local lists below 
# Manually select which records should be part of the list and insert into list below

# Set field name variable depending on county
while True:
    try:
        if UserCountyResponse.lower() == "carson":
            field_name = ["Assessed_Owner"]
            fed_whereclause = "Assessed_Owner LIKE '%FOREST%' OR Assessed_Owner LIKE '%POSTAL%' OR Assessed_Owner LIKE '%UNITED%' OR Assessed_Owner LIKE '%GUARD%' OR Assessed_Owner LIKE '%RANGER%' OR Assessed_Owner LIKE '%DEPT%' OR Assessed_Owner LIKE '%BUREAU%' OR Assessed_Owner LIKE '%VETERANS%'"
            state_whereclause = "Assessed_Owner LIKE '%CONSERVANCY%' OR Assessed_Owner LIKE '%NEVADA%' OR Assessed_Owner LIKE '%CALIFORNIA%' OR Assessed_Owner LIKE '%UNIV%' OR Assessed_Owner LIKE '%UNIVERSITY%'"
            local_whereclause = "Assessed_Owner LIKE '%DOUGLAS%' OR Assessed_Owner LIKE '%EL DORADO%' OR Assessed_Owner LIKE '%WASHOE%' OR Assessed_Owner LIKE '%PLACER%' OR Assessed_Owner LIKE '%CARSON CITY%' OR Assessed_Owner LIKE '%UNIFIED%' OR Assessed_Owner LIKE '%IMPROVEMENT%' OR Assessed_Owner LIKE '%DIST%' OR Assessed_Owner LIKE '%COUNTY%' OR Assessed_Owner LIKE '%SOUTH TAHOE%' OR Assessed_Owner LIKE '%FIRE%'"

            break
        elif UserCountyResponse.lower() == "douglas":
            field_name = ["PANAME"]
            fed_whereclause = "PANAME LIKE '%FOREST%' OR PANAME LIKE '%POSTAL%' OR PANAME LIKE '%UNITED%' OR PANAME LIKE '%GUARD%' OR PANAME LIKE '%RANGER%' OR PANAME LIKE '%DEPT%' OR PANAME LIKE '%BUREAU%' OR PANAME LIKE '%VETERANS%'"
            state_whereclause = "PANAME LIKE '%CONSERVANCY%' OR PANAME LIKE '%NEVADA%' OR PANAME LIKE '%CALIFORNIA%' OR PANAME LIKE '%UNIV%' OR PANAME LIKE '%UNIVERSITY%'"
            local_whereclause = "PANAME LIKE '%DOUGLAS%' OR PANAME LIKE '%EL DORADO%' OR PANAME LIKE '%WASHOE%' OR PANAME LIKE '%PLACER%' OR PANAME LIKE '%CARSON CITY%' OR PANAME LIKE '%UNIFIED%' OR PANAME LIKE '%IMPROVEMENT%' OR PANAME LIKE '%DIST%' OR PANAME LIKE '%COUNTY%' OR PANAME LIKE '%SOUTH TAHOE%' OR PANAME LIKE '%FIRE%'"

            break
        elif UserCountyResponse.lower() == "el_dorado" or UserCountyResponse.lower() == "el dorado":
            field_name = ["OWNER_NAME"]
            fed_whereclause = "OWNER_NAME LIKE '%FOREST%' OR OWNER_NAME LIKE '%POSTAL%' OR OWNER_NAME LIKE '%UNITED%' OR OWNER_NAME LIKE '%GUARD%' OR OWNER_NAME LIKE '%RANGER%' OR OWNER_NAME LIKE '%DEPT%' OR OWNER_NAME LIKE '%BUREAU%' OR OWNER_NAME LIKE '%VETERANS%'"
            state_whereclause = "OWNER_NAME LIKE '%CONSERVANCY%' OR OWNER_NAME LIKE '%NEVADA%' OR OWNER_NAME LIKE '%CALIFORNIA%' OR OWNER_NAME LIKE '%UNIV%' OR OWNER_NAME LIKE '%UNIVERSITY%'"
            local_whereclause = "OWNER_NAME LIKE '%DOUGLAS%' OR OWNER_NAME LIKE '%EL DORADO%' OR OWNER_NAME LIKE '%WASHOE%' OR OWNER_NAME LIKE '%PLACER%' OR OWNER_NAME LIKE '%CARSON CITY%' OR OWNER_NAME LIKE '%UNIFIED%' OR OWNER_NAME LIKE '%IMPROVEMENT%' OR OWNER_NAME LIKE '%DIST%' OR OWNER_NAME LIKE '%COUNTY%' OR OWNER_NAME LIKE '%SOUTH TAHOE%' OR OWNER_NAME LIKE '%FIRE%'"

            break
        elif UserCountyResponse.lower() == "placer":
            field_name = ["OWNER1"]
            fed_whereclause = "OWNER1 LIKE '%FOREST%' OR OWNER1 LIKE '%POSTAL%' OR OWNER1 LIKE '%UNITED%' OR OWNER1 LIKE '%GUARD%' OR OWNER1 LIKE '%RANGER%' OR OWNER1 LIKE '%DEPT%' OR OWNER1 LIKE '%BUREAU%' OR OWNER1 LIKE '%VETERANS%'"
            state_whereclause = "OWNER1 LIKE '%CONSERVANCY%' OR OWNER1 LIKE '%NEVADA%' OR OWNER1 LIKE '%CALIFORNIA%' OR OWNER1 LIKE '%UNIV%' OR OWNER1 LIKE '%UNIVERSITY%'"
            local_whereclause = "OWNER1 LIKE '%DOUGLAS%' OR OWNER1 LIKE '%EL DORADO%' OR OWNER1 LIKE '%WASHOE%' OR OWNER1 LIKE '%PLACER%' OR OWNER1 LIKE '%CARSON CITY%' OR OWNER1 LIKE '%UNIFIED%' OR OWNER1 LIKE '%IMPROVEMENT%' OR OWNER1 LIKE '%DIST%' OR OWNER1 LIKE '%COUNTY%' OR OWNER1 LIKE '%SOUTH TAHOE%' OR OWNER1 LIKE '%FIRE%'"

            break
        elif UserCountyResponse.lower() == "washoe":
            field_name = ["LASTNAME"]
            fed_whereclause = "LASTNAME LIKE '%FOREST%' OR LASTNAME LIKE '%POSTAL%' OR LASTNAME LIKE '%UNITED%' OR LASTNAME LIKE '%GUARD%' OR LASTNAME LIKE '%RANGER%' OR LASTNAME LIKE '%DEPT%' OR LASTNAME LIKE '%BUREAU%' OR LASTNAME LIKE '%VETERANS%'"
            state_whereclause = "LASTNAME LIKE '%CONSERVANCY%' OR LASTNAME LIKE '%NEVADA%' OR LASTNAME LIKE '%CALIFORNIA%' OR LASTNAME LIKE '%UNIV%' OR LASTNAME LIKE '%UNIVERSITY%'"
            local_whereclause = "LASTNAME LIKE '%DOUGLAS%' OR LASTNAME LIKE '%EL DORADO%' OR LASTNAME LIKE '%WASHOE%' OR LASTNAME LIKE '%PLACER%' OR LASTNAME LIKE '%CARSON CITY%' OR LASTNAME LIKE '%UNIFIED%' OR LASTNAME LIKE '%IMPROVEMENT%' OR LASTNAME LIKE '%DIST%' OR LASTNAME LIKE '%COUNTY%' OR LASTNAME LIKE '%SOUTH TAHOE%' OR LASTNAME LIKE '%FIRE%'"

            break
        else:
            print ("County entered is not valid. Try a county listed below")
            for x in County_List:
                print (x)
            print('\n')
            continue
    except Exception as e:
        print (e)
        sys.exit()

# Searchcursor to create federal owner list
print("Federal List:")
with arcpy.da.SearchCursor(parcel_out_data, field_name, fed_whereclause) as cursor:
    fed_list = sorted({row[0] for row in cursor})  
nospace_fedlist = [x.strip(' ') for x in fed_list]
print(nospace_fedlist)

# Searchcursor to create state owner list
print("State List:")
with arcpy.da.SearchCursor(parcel_out_data, field_name, state_whereclause) as cursor:
    state_list = sorted({row[0] for row in cursor})
nospace_statelist = [x.strip(' ') for x in state_list]
print(nospace_statelist)

# Searchcursor to create local owner list
print("Local List:")
with arcpy.da.SearchCursor(parcel_out_data, field_name, local_whereclause) as cursor:
    local_list = sorted({row[0] for row in cursor})
nospace_locallist = [x.strip(' ') for x in local_list]
print(nospace_locallist)

Federal List:
['9071 FOREST DRIVE CABIN CA LLC', 'AL TAHOE FOREST HOMES ASSN', 'BERKELEY PLAYGROUND COMM & REC & PARKS DEPT', 'BUREAU OF LAND MANAGEMENT', 'BUTCHER CARYN L GUARDIAN & CHEYNE P ESTATE MINOR', 'CA STATE DEPT TRANSPORTATION', 'CALIFORNIA STATE OF & DEPT GEN SERVICES REAL ESTAT', 'COUNTY OF EL DORADO & DEPT OF PUBLIC WORKS', 'COUNTY OF EL DORADO & DEPT OF TRANSPORTATION', 'DE NAULT LINDA GUARDIAN OF & NAULT BENJAMIN IV', 'DEPT OF VETERANS AFFAIRS  & ERSKINE NEIL H TR', 'DEPT OF VETERANS AFFAIRS  & MASTERS DANE C', 'DEPT OF VETERANS AFFAIRS  & SLEZAK FRANK J CO TR', 'DEPT OF VETERANS AFFAIRS & RIVES DONALD E JR', 'DEPT OF VETERANS AFFAIRS & WILLIAMS MATTHEW G', 'DEPT OF VETRANS AFFAIRS & WILSON VIVIAN M', 'FOREST INVESTMENTS A CA LLC', 'GARCIA JOSEPH A ESTATE & EDC PUBLIC GUARDIAN CSVTR', 'GIUSTINA ROBERT SUC TR & FOREST S FAM TR OF 9/29/0', 'GUARDINO RICHARD V TR & LORI ANNE TR', 'LAKE VALLEY RANGER STA & U S FOREST SERVICE', 'SAFFORD HUGH DEFOREST TR & MARY KELLY TR', 'STA

In [20]:
fedOwnList = ("USA FOREST SERVICE", "USDA FOREST SERVICE", "USDA - FOREST SERVICE", "UNITED STATES POSTAL", 
              "UNITED STATES OF AMERICA", "UNITED STATES FOREST SERVICE", "U S POSTAL SERVICE", "U S COAST GUARD",
              "U S A FOREST SERVICE``", "U S A FOREST SERVICE", "LAKE VALLEY RANGER STA", "DEPT OF VETRANS AFFAIRS%", 
              "DEPT OF VETERANS AFFAIRS%", "DEPT OF VETERANS AFFAIRS %", "BUREAU OF LAND MANAGEMENT", "U S FOREST SERVICE",
              "DEPT OF VETERANS AFFAIRS  & ERSKINE NEIL H TR", "DEPT OF VETERANS AFFAIRS  & MASTERS DANE C", 
              "DEPT OF VETERANS AFFAIRS  & SLEZAK FRANK J CO TR", "DEPT OF VETERANS AFFAIRS & RIVES DONALD E JR"
              "DEPT OF VETERANS AFFAIRS & WILLIAMS MATTHEW G DBA WILLIAMS VACATION HOME", "DEPT OF VETRANS AFFAIRS & WILSON VIVIAN M",
              "DEPARTMENT OF TRANSPORTATION", "USA FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S A FOREST SERVICE & LAKE TAHOE BASIN MNGMT UNIT",
              "U S A FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S D A FOREST SERVICE", "UNITED STATES & DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & ATTN RICHARD T FLYNN", "UNITED STATES OF AMERICA & DEPARTMENT OF AGRICULTU", "UNITED STATES OF AMERICA & F/S DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & FOREST SER. DEPT OF AG.", "UNITED STATES OF AMERICA & FOREST SERVICE", "UNITED STATES OF AMERICA & FOREST SERVICE (USDA)",
              "UNITED STATES OF AMERICA & FOREST SERVICE DEPT OF", "UNITED STATES OF AMERICA & FOREST SERVICE TAHOE BA", "UNITED STATES OF AMERICA & FOREST SERVICE USDA",
              "UNITED STATES OF AMERICA & FOREST SVC/DEPT OF AGRI", "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MANAGM",
              "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MGT UN", "UNITED STATES OF AMERICA & REGIONAL LAND ADJUSTMEN",
              "UNITED STATES OF AMERICA & U S FOREST SERVICE", "UNITED STATES OF AMERICA & U S FOREST SERVIE",
              "UNITED STATES OF AMERICA & USDA FOREST SER LAKE TA", "UNITED STATES OF AMERICA & USDA FOREST SERVICE")

stateOwnList = ("TAHOE CONSERVANCY", "STATE OF NEVADA FOREST SERVICE", "STATE OF NEVADA", "STATE OF CALIFORNIA THE", 
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "STATE OF CA", "REGENTS OF UNIV OF CALIF",
                "UNIVERSITY CALIFORNIA REGENTS", "UNIVERSITY OF NEVADA RENO", "NEVADA, STATE OF", "NEVADA STATE OF", 
                "CALIFORNIA TAHOE CONSERVANCY ET AL", "CALIFORNIA TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "CALIFORNIA STATE OF ET AL", "CALIFORNIA STATE OF", "CA STATE DEPT TRANSPORTATION", 
                "CA TAHOE CONSERVANCY", "CALIFORNIA STATE OF TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "NEVADA DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY", "STATE OF CALIFORIA & CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CA TAHOE CONSERVANCY", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSEVANCY", "STATE OF CALIFORNIA & DEPART OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPARTMENT OF GENERAL SERVIC", 
                "STATE OF CALIFORNIA & DEPARTMENT OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPT OF GEN SRVS R E DIV", "STATE OF CALIFORNIA & DEPT OF GENERAL SERVICES",
                "STATE OF CALIFORNIA & DEPT OF PARKS & RECREATION", "STATE OF CALIFORNIA & DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & PARKS & RECREATION",
                "STATE OF CALIFORNIA (EASEMENT) & CALIFORNIA TAHOE")

localOwnList = ("ZEPHYR COVE GENERAL IMP DIST", "WASHOE COUNTY SCHOOL DISTRICT BOARD", "WASHOE COUNTY", "WASHOE TRIBE OF NV & CA", 
                "TALMONT RESORT IMPROVEMENT DISTRICT", "TALMONT RESORT IMPR DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMP DIST", "TAHOE PARADISE RESORT IMP DIST", "TAHOE PARADISE RES IMP DST",
                "TAHOE FOREST HOSPITAL DISTRICT", "TAHOE TRUCKEE UNIFIED SCHOOL DISTRICT", "TAHOE TRUCKEE UNIFIED SCH DIST", 
                "TAHOE DOUGLAS FIRE PROTECT DIST", "TAHOE DOUGLAS SEWER DIST", "TAHOE DOUGLAS DISTRICT", 
                "TAHOE CITY PUBLIC UTILITY DISTRICT", "TAHOE CITY PUBLIC UTILITY DIST", "TAHOE CITY PUBLIC UTILDIST", 
                "TAHOE CITY PUB UTILITY DST", "TAHOE CITY PUB UTILITY DIS", "TAHOE CITY P U D", "TAHOE CITY CEMETERY DIST", 
                "SOUTH TAHOE REDEVELP AGENCY", "SOUTH TAHOE REFUSE CO", "SOUTH TAHOE PUD", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTH TAHOE PUBLIC UTILITYDIST", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTILITY DIS", 
                "SOUTH TAHOE PUBLIC UTILITY", "SOUTH TAHOE PUBLIC UTIL DT", "SOUTH TAHOE PUBLIC UTIL DIST", 
                "SOUTH TAHOE PUBLIC", "SOUTH TAHOE PUB UTIL DIST", "SOUTH LAKE TAHOE CTYOF 1/3", "SOUTH LAKE TAHOE CITY OF", 
                "SO TAHOE PUBLIC UTILITY DIST", "SO TAHOE PUB UTIL DIST", "SIERRA NEVADA COLLEGE", "ROUND HILL GEN IMP DIST",
                "PLACER COUNTY REDEVELOPMENT AGENCY", "PLACER COUNTY OF", "PLACER COUNTY", "NORTH TAHOE PUBLIC UTL DIST",
                "NORTH TAHOE PUBLIC UTILITY DISTRICT", "NORTH TAHOE PUBLIC UTILITY DIST", "NORTH TAHOE PUBLIC UTILITY DIS", 
                "NORTH TAHOE PUBLIC UTILITIES DIST", "NORTH TAHOE PUBLIC UTILIITY DISTRICT", "NORTH TAHOE P U D",
                "NORTH TAHOE FIRE PROTECTION DISTRICT", "NORTH TAHOE FIRE PROTECTION", "NORTH TAHOE FIRE DIST",
                "NORTH LAKE TAHOE FIRE PROTECTION DIST", "N TAHOE FIRE PROTECTION DIST", "MEEKS BAY FIRE PROT DIST", 
                "LAKERIDGE GENERAL IMP DIST", "LAKE VALLEY FIRE PROTECTION", "LAKE VALLEY FIRE PROT DST", "LAKE VALLEY FIRE PROT DIST", 
                "LAKE VALLEY FIRE DISTRICT", "LAKE TAHOE UNIFIED SCHOOL DIST", "LAKE TAHOE SCHOOL", "LAKERIDGE GENERAL IMP DIST", 
                "LAKE TAHOE FIRE PROTECTION DIST", "LAKE TAHOE FIRE PROTECT DIST", "LAKE TAHOE COMM COLLEGE DIST",
                "LAKE TAHOE COMM COL DIST", "KINGSBURY GENERAL IMP DISTRICT", "KINGSBURY GENERAL IMP DIST",
                "INCLINE VILLAGE GENERAL IMPROVEMENT DISTRICT", "INCLINE VILLAGE GENERAL IMPROVEMENT DIST", 
                "DOUGLAS COUNTY SEWER DIST", "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS COUNTY", "DOUGLAS CO SEWER IMP DIST #1", 
                "COUNTY OF EL DORADO", "CITY OF SOUTH LAKE TAHOE", "EL DORADO IRRIGATION DISTRICT", 
                "HAPPY HOMESTEAD CEMETERY DIST", "WASHOE TRIBE", "SOUTHTAHOE PUBLIC UTILITY DIST", "DOUGLAS COUNTY TRUSTEE", 
                "DOUGLAS COUNTY TRUSTEE (HOLD)", "WASHOE TRIBE OF NEVADA AND CALIFORNIA", "ALPINE SPRINGS CO WATER DIST", 
                "ALPINE SPRINGS COUNTY WATER DISTRICT", "ALPINE SPRINGS WATER DISTRICT", "NORTHSTAR COMMUNITY SERVICE DISTRICT",
                "SQUAW VALLEY CO WATER DIST", "SQUAW VALLEY PUBLIC SERVICE DISTRICT", "TRUCKEE TAHOE AIRPORT DISTRICT", 
                "COUNTY OF EL DORADO & ATTEN: PAUL MCINTOSH", "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", 
                "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & C/O BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & COUNSEL",
                "COUNTY OF EL DORADO & COUNSEL'S OFFICE", "COUNTY OF EL DORADO & DEPARTMENT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPARTMENT OF TRANSPORTATION",
                "COUNTY OF EL DORADO & DEPT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPT OF TRANSPORTATION", "COUNTY OF EL DORADO & GENERAL SERVICES DEPARTMENT",
                "COUNTY OF EL DORADO & OF EL DORADO", "COUNTY OF EL DORADO & PUBLIC WORKS DEPARTMENT", "EL DORADO CO OFFICE EDUCATION", "EL DORADO COUNTY & SUPERINTENDENT OF SCHOOLS",
                "EL DORADO COUNTY & BOARD OF SUPERVISORS", "LAKE TAHOE COMMUNITY COLLEGE DIST", "LAKE VALLEY RANGER STA & U S FOREST SERVICE",
                "LAKE VALLEY FIRE PROTECTION & DISTRICT POLITICAL S", "SOUTH TAHOE PUBLIC & UTILITY DISTRICT",
                "SOUTH TAHOE PUBLIC UTILITY &  DISTRIC", "SOUTH TAHOE PUBLIC UTIL DIST & CA MUNICIPAL CORP", "TAHOE CITY PUBLIC UTIL DST",
                "TAHOE RESOURCE CONSERVATION &  DISTRIC", "TAHOE RESOURCE CONSERVATION DIST  C/O DISTRICT MANAGER", "FALLEN LEAF COMM SERVICES DIST",
                "FALLEN LEAF LAKE COMM SERVDIST") 

ownerListVal = ("STATE OF NEVADA", "STATE OF NEVADA FOREST SERVICE", "NEVADA, STATE OF",
                "NEVADA, DEPT OF TRANSPORTATION",
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "CALIFORNIA STATE OF", "STATE OF CALIFORINA",
                "STATE OF CALIFORIA", "STATE OF CA", "CALIFORNIA STATE OF ET AL",
                "UNITED STATES OF AMERICA", "U S A FOREST SERVICE", "U S A  FOREST SERVICE", "USA FOREST SERVICE",
                "U S COAST GUARD", "BUREAU OF LAND MANAGEMENT", "DEPT OF VETRANS AFFAIRS", "DEPT OF VETERANS AFFAIRS",
                "DEPT OF VETERANS AFFAIRS %",
                "U S FOREST SERVICE", "UNITED STATES FOREST SERVICE", "UNITED STATES POSTAL",
                "CARSON CITY", "DOUGLAS COUNTY",
                "CITY OF SOUTH LAKE TAHOE", "COUNTY OF EL DORADO", "EL DORADO COUNTY OF", "PLACER COUNTY",
                "PLACER COUNTY OF", "PLACER COUNTY REDEVELOPMENT AGENCY",
                "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS CO SEWER IMP DIST #1", "LAKE TAHOE UNIFIED SCHOOL DIST",
                "TALMONT RESORT IMP DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMPROVEMENT DISTRICT", "CALIFORNIA TAHOE CONSERVANCY", "CA TAHOE CONSERVANCY",
                "LAKE TAHOE COMM COL DIST",
                "SO TAHOE PUB UTIL DIST", "SO TAHOE PUBLIC UTILITY DIST", "SOUTH TAHOE PUB UTIL DIST",
                "SOUTH TAHOE PUBLIC UTIL DIST", "SOUTH TAHOE PUBLIC UTIL DT",
                "SOUTH TAHOE PUBLIC UTILITY DIS", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTHTAHOE PUBLIC UTILITY DIST", "CALIFORNIA TAHOE CONSERVANCY ET AL")


#### List all field names and their index number

In [14]:
field_names = [field.name for field in arcpy.ListFields(parcel_out_data)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

0 OBJECTID_1
1 Shape
2 OBJECTID
3 POLY_ID
4 PRCL_ID
5 POLY_CLASS
6 POLY_CREAT
7 POLY_DESCR
8 EXCPTN_LIT
9 TAX_CD
10 TAXCDDESCR
11 USECD_1
12 USECDCL_1
13 USECDTY_1
14 USECDLIT_1
15 USECD_2
16 USECDCL_2
17 USECDTY_2
18 USECDLIT_2
19 ACREAGE
20 LEGAL_DESC
21 SUBDIV_LIT
22 NEIGHBORHD
23 ADDRSTNBR
24 HOUSE_SFX
25 ADDRSTDIR
26 ADDRSTPRFX
27 ADDRSTNAME
28 ADDRSTTYPE
29 ADDRSTPOST
30 ADDRUNITTY
31 ADDRUNITNB
32 ADDRFLOOR
33 PRCL_ADDR
34 STRUCT_VAL
35 LAND_VAL
36 YR_BUILT
37 WATERSRCCD
38 SEWERCD
39 DWELLUNITS
40 BEDROOMS
41 IMPR_SQ_FT
42 HO_XMPT
43 OWNER_NAME
44 MAIL_ADDR1
45 MAIL_ADDR2
46 MAIL_ADDR3
47 MAIL_ADDR4
48 TRA_PRI
49 TRA_SEC
50 JURS_LIT
51 SUPR_LIT
52 SCHL_DIST
53 FIRE_DIST
54 WATER_DIST
55 AREA
56 PERIMETER
57 Shape_Length
58 Shape_Area
59 APN_NEW
60 PPNO_NEW
61 HSE_NUMBR_NEW
62 UNIT_NUMBR_NEW
63 STR_DIR_NEW
64 STR_NAME_NEW
65 STR_SUFFIX_NEW
66 APO_ADDRESS_NEW
67 PSTL_TOWN_NEW
68 PSTL_STATE_NEW
69 PSTL_ZIP5_NEW
70 OWN_FIRST_NEW
71 OWN_LAST_NEW
72 OWN_FULL_NEW
73 MAIL_ADD1_NEW
74 M

#### City of South Lake Tahoe script that seperates El Dorado's adress into separate fields

In [15]:
######IMPORTANT#########
# TEST!!!!!!
import arcpy
import sys
from arcpy import env
import re

# Set up variable for the existing parcel FC and the path to unzip to.
fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\El_Dorado\\Modified\\EL_Staging.gdb\\Parcels\\El_Dorado_Parcels" 

# Set up the regex queries for the data.
cityStateZipRegex = r'(.+?)\s([A-Z]{1,2})\s(?=\d)(.*)'
poBoxRegex = r'([^x]+)\W(P\s*O BOX\W*[0-9]{1,6})'
addressRegex = r'(\d{1,5}\D+.+)'
canadaRegex = r'(.+?)\s([A-Z]{1,2})\s(CANADA)\s(.*)'
brazilRegex = r'(.+?)\s(BRAZIL)\s(.*)'
    
# Set up list for addresses with a country name in the mail_addr4 column.
countriesList = ['japan']

# Add extra fields to calculate values
arcpy.AddField_management(fc, "ADDRESS2", "TEXT", 250)
arcpy.AddField_management(fc, "COUNTRY_NEW", "TEXT", 250)
arcpy.AddField_management(fc, "ZIP_NEW", "TEXT", 250)
arcpy.AddField_management(fc, "OWN_NEW", "TEXT", 250)
# print ("Fields Added.")
    
# Use Update Cursor to loop through each record and using REGEX parse out owner and address info.
with arcpy.da.UpdateCursor(fc, ['OWNER_NAME','MAIL_ADDR1','MAIL_ADDR2','MAIL_ADDR3','MAIL_ADDR4','OWN_NEW','MAIL_ADD1_NEW','MAIL_CITY_NEW','MAIL_STATE_NEW','ZIP_NEW','COUNTRY_NEW','PRCL_ID']) as cursor:
    print('Parsing ownership and address info...')
    for row in cursor:
        # Start from mail_addr4 and work left.
        if row[4] != ' ':
            if row[4] != 'UNKNOWN' and row[4].lower() not in countriesList:
                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, str(row[4]))
                if cityStateZip is not None:
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)
                    country = ''
                else:
                    continue
                # Check to see if address starts with PO Box and assign variable.
                if str(row[3]).startswith('PO') or str(row[3]).startswith('P O'):
                    address = str(row[3])
                elif "PO BOX" in str(row[3]) or "P O BOX" in str(row[3]) or "P.O. BOX" in str(row[3]):
                    address = str(row[3])

                # Parse out address that doesn't have PO Box and assign variable.
                else:
                    add = re.search(addressRegex,str(row[3]))
                    address = add.group(1)

                # Assign owner variable.
                owner = str(row[0])+' '+str(row[1])+' '+str(row[2])
            elif row[4].lower() in countriesList:
                country = str(row[4])
                state = str(row[3])
                city = str(row[2])
                address = str(row[1])
                owner = str(row[0])
                zipCode = ''
            else:
                owner = str(row[0])
                address = ''
                city = ''
                state = ''
                zipCode = ''
                country = ''

        # If mail_addr4 is "empty".
        elif row[3] != ' ':
            print("Working on MAIL_ADDR3")
            # Parse out city, state, and zip code.
            cityStateZip = re.search(cityStateZipRegex, str(row[3]))

            # Foreign addresses won't parse so assign country, owner, address, and city variables. Set state and zip to blanks.
            if cityStateZip is None:
                country = str(row[3])
                owner = str(row[0])
                address = str(row[1])
                city = str(row[2])
                state = ''
                zipCode = ''
            else:
                country = ''
                row2 = str(row[2])

                # Sanitize rows that start with a space.
                if str(row[2]).startswith(' '):
                    row2 = str(row[2])[1:]

                # Parse out city, state, and zip code and assign variables.
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)

                # Check to see if address starts with PO Box and assign variable.
                if row2.startswith('PO') or row2.startswith('P O') or row2.startswith('P.O.'):
                    address = row2

                # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                elif "PO BOX" in row2 or "P O BOX" in row2 or row2.startswith('ONE ') or row2.startswith('TWO '):
                    address = row2
                else:
                    # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                    add = re.search(addressRegex,row2)
                    if add is None:
                        address = 'None'
                    else:
                        address = add.group(1)

                # Assign owner variable.
                owner = str(row[0])+' '+str(row[1])

        # Before moving to mail_addr2 must capture "blanks" and USA owned parcels and insert blanks.
        elif row[0] == 'UNITED STATES OF AMERICA':
            cityStateZip = re.search(cityStateZipRegex, str(row[2]))
            owner = str(row[0])
            address = str(row[1])
            if cityStateZip is None:
                city = ''
                state = ''
                zipCode = ''
            else:
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)
            country = ''
        elif row[0] == ' ':
            owner = ''
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''
        elif row[1] == ' ':
            owner = str(row[0])
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''

        # Parce the rest of the address info.
        else:
            print("Working on MAIL_ADDR2")
            if str(row[2]) == ' ':
                owner = str(row[0])
                address = str(row[1])
                city = ''
                state = ''
                zipCode = ''
                country = ''
            else:
                row2 = str(row[2])

                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, row2)

                # if it can't parse it's a foreign address and assign country variable.
                if cityStateZip is None:
                    if "CANADA" in row2:
                        cityStateZip = re.search(canadaRegex, row2)
                        city = cityStateZip.group(1)
                        state = cityStateZip.group(2)
                        zipCode = cityStateZip.group(4)
                        country = cityStateZip.group(3)
                    if "BRAZIL" in row2:
                        cityStateZip = re.search(brazilRegex, row2)
                        city = cityStateZip.group(1)
                        state = ''
                        zipCode = cityStateZip.group(3)
                        country = cityStateZip.group(2)
                else:
                    row1 = str(row[1])
                    country = ''
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)

                    # Sanitize rows that start with a space.
                    if row1.startswith(' '):
                        row1 = row1[1:]

                    # Check to see if address starts with PO Box and assign variable.
                    if row1.startswith('PO') or row1.startswith('P.O.') or row1.startswith('P O') or row1.startswith('P  O'):
                        address = str(row[1])

                    # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                    elif "PO BOX" in row1 or "P O BOX" in row1:
                        poBox = re.search(poBoxRegex,row1)

                        # If it can't be parsed assign variable.
                        if poBox is None:
                            address = row1
                        else:
                            address = poBox.group(2)
                    else:
                        # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                        add = re.search(addressRegex,row1)

                        # Have exception for addresses that spell out 'one' instead of '1'.
                        if add is None or row1.startswith('ONE'):
                            address = row1
                        else:
                            address = add.group(1)

                # Set owner variable.
                owner = str(row[0])

        # Set up columns for the UpdateRow function.
        row[5] = owner
        row[6] = address
        row[7] = city
        row[8] = state
        row[9] = zipCode
        row[10] = country

        # Update the row.
        cursor.updateRow(row)
print("Completed!")

Parsing ownership and address info...
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

...

Working on MAIL_ADDR3
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR3
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on MAIL_ADDR2
Working on

In [17]:
fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\El_Dorado\\Modified\\EL_Staging.gdb\\Parcels\\El_Dorado_Parcels" 
fieldnames = ['OWN_NEW', 'ZIP_NEW','COUNTRY_NEW', 'OWN_FULL_NEW', 'MAIL_ZIP5_NEW', 'MAIL_CITY_NEW']
with arcpy.da.UpdateCursor(fc, fieldnames) as cursor:
    for row in cursor:
        own = row[0]
        if not(own is None or "C/O LINCHRIS HOTEL" in own):
            row[3] = own
        elif ("C/O LINCHRIS HOTEL" in own):
            row[3] = "LCOF LAKE TAHOE INVEST LLC DE LLC"
        else:
            row[3] = ""
        # make zip code 5 digits
        zipcode = row[1]
        if not (zipcode is None):
            row[4] = zipcode[:5]
        else:
            row[4] = ""
        # put countries in city field
        country = row[2]
        if not (country is None or country == ""):
            if not ("BEDFORSHIRE" or "JAPAN" in country):
                row[5] = "".join([char for char in country if not char.isdigit()]).strip()
            elif ("JAPAN" in country):
                row[5] = "JAPAN"
            elif ("BEDFORSHIRE" in country):
                row[5] = "BEDFORSHIRE"
        # Update the row.
        cursor.updateRow(row)
print("Completed!")

Completed!


In [18]:
# Set up variable for the existing parcel FC and the path to unzip to.
fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\El_Dorado\\Modified\\EL_Staging.gdb\\Parcels\\El_Dorado_Parcels" 
# ElDorado = "C:\\GIS\\ParcelUpdate\\2020_11\\El_Dorado\\Modified\\EL_Staging.gdb\\Parcels\\El_Dorado_Parcels"
# Join = "C:\\GIS\\ParcelUpdate\\2020_11\\El_Dorado\\Modified\\EL_Staging.gdb\\Join_Delete"
# joinfield = prcl_id

# # In memory layers
# wk_memory = "in_memory" + "\\"
# ElDoradoJoin = wk_memory + "\\ElDoradoJoin"


# Use update cursor to populate the address2 field with C/o, ATTN and DBA values
field_names = ['MAIL_ADDR1', 'MAIL_ADDR2', 'MAIL_ADDR3', 'MAIL_ADDR4', 'ADDRESS2']
with arcpy.da.UpdateCursor(fc, field_names) as cursor:
        for row in cursor:
            # set apn
            mail1 = row[0]
            mail2 = row[1]
            mail3 = row[2]
            mail4 = row[3]
            address2 = row[4]
            if ('DBA' in mail1 or '/' in mail1 or 'ATTN' in mail1):
                row[4] = mail1
            elif ('DBA' in mail2 or '/' in mail2 or 'ATTN' in mail2):
                if not (address2 is None):
                    row[4] = mail1 + "," + " " + mail2
                else:
                    row[4] = mail2
            elif ('DBA' in mail3 or '/' in mail3 or 'ATTN' in mail3):
                print("mail3 has address")
            elif ('DBA' in mail4 or '/' in mail4 or 'ATTN' in mail4):
                print("mail4 has address")
            else:
                row[4] = ""
            # update all rows
            cursor.updateRow(row)
print("Address2 field complete")

Address2 field complete


### Translate County Values to TRPA Values and Fields

In [22]:
# process Washoe County fields
if UserCountyResponse == "Washoe":
    print ("Processing the {} dataset".format(County_List[4]))
    field_names = ['PIN', 'APN_NEW', 'APN', 'PPNO_NEW',
                   'FullAddres','HSE_NUMBR_NEW','UNIT_NUMBR_NEW', 'STREETDIR', 'STR_DIR_NEW', 
                   'STREET', 'STR_NAME_NEW', 'STR_SUFFIX_NEW','PSTL_STATE_NEW',
                   'FIRSTNAME', 'OWN_FIRST_NEW', 'LASTNAME', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'MAILING1','MAIL_ADD1_NEW', 'MAILING2','MAIL_ADD2_NEW',
                   'MAILCITY', 'MAIL_CITY_NEW', 'MAILSTATE','MAIL_STATE_NEW', 
                   'MAILZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE',
                   'LAND_USE', 'COUNTY_LANDUSE_CODE', 
                   'LANDASS', 'AS_LANDVALUE_NEW','BUILDASS','AS_IMPROVALUE_NEW','AS_SUM_NEW',
                   'LANDAPR','TAX_LANDVALUE_NEW','BUILDAPR','TAX_IMPROVALUE_NEW','TAX_SUM_NEW',
                   'TAX_YEAR_NEW', 'UNITS', 'UNITS_NEW', 'BEDROOMS', 'BEDROOMS_NEW', 'BATHS', 
                   'BATHROOMS_NEW', 'YEARBLT', 'YEAR_BUILT_NEW', 'SQFEET', 'BUILDING_SQFT_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True):
                row[1] = apn
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not ppno is None:
                row[3] = int(ppno)
            else:
                row[3] = ""
            # set house number
            fulladdress = row[4]
            if not (fulladdress is None or fulladdress == ""):
                if fulladdress[0].isdigit():
                    row[5] = (fulladdress.rsplit(' ')[0].strip())
                else:
                    row[5] = ""
            else:
                row[5] = 0
            # set unit number
            if not (fulladdress is None or fulladdress == ""):
                if fulladdress[-1].isdigit():
                    if not ('STATE ROUTE 28' in fulladdress):
                        row[6] = "#" + (fulladdress.rsplit(' ')[-1].strip())
                    else:
                        if not (fulladdress.rsplit(' ')[-1] == '28'):
                            row[6] = "#" + (fulladdress.rsplit(' ')[-1].strip())
                        else:
                            if not ('STATE ROUTE 28 28' in fulladdress): 
                                row[6] = ""
                            else:
                                row[6] = "#" + (fulladdress.rsplit(' ')[-1].strip())
                else:
                    if not ('US HIGHWAY 395' in fulladdress):
                        if len(fulladdress.rsplit(' ')[-1]) == 1:
                            row[6] = "#" + (fulladdress.rsplit(' ')[-1].strip())
                        elif not (len(fulladdress.rsplit(' ')[-1]) == 1):
                            if fulladdress[-2].isdigit():
                                row[6] = "#" + (fulladdress.rsplit(' ')[-1].strip())
                            else:
                                row[6] = ""
                        else:
                            row[6] = ""
                    else:
                        row[6] = ""
            else:
                row[6] = ""
            # set street direction
            stdir = row[7]
            if not (stdir is None):
                row[8] = (stdir.strip())
            else:
                row[8] = ""
            # set street name    
            stname = row[9]
            if (stname is None or stname == "" or stname.isspace() == True or 'US HIGHWAY 395' in stname):
                if 'US HIGHWAY 395' in stname:
                    row[10] = (fulladdress.rsplit(' ')[1].strip()) + " " + (fulladdress.rsplit(' ')[2].strip()) + " " + (fulladdress.rsplit(' ')[3].strip()) + " " + (fulladdress.rsplit(' ')[4].strip())
                elif not ('US HIGHWAY 395' in stname):
                    if fulladdress[0].isdigit():
                        if len(fulladdress.split()) == 3:
                            row[10] = (fulladdress.rsplit(' ')[1].strip())
                        elif len(fulladdress.split()) == 4:
                            row[10] = (fulladdress.rsplit(' ')[1].strip()) + " " + (fulladdress.rsplit(' ')[2].strip())
                        else:
                            print("Error")
                    else:
                        row[10] = (fulladdress.strip())
                else:
                    row[10] = ""
            elif not (stname is None or stname == "" or stname.isspace() == True or 'US HIGHWAY 395' in stname):
                if not (stname in ('CROSS BOW', 'ENTERPRISE','STATE ROUTE 28', 'UNSPECIFIED', '')):
                    row[10] = (stname.rsplit(" ", 1)[0].strip())
                elif stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', ''):
                    row[10] = (stname.strip())
                else:
                    row[10] = ""    
            else:
                row[10] = ""
            # set street suffix
            stname = row[9]
            if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395', ''):
                if not (stname is None or stname == "" or stname.isspace() == True):
                    row[11] = (stname.rsplit(' ')[-1].strip())
                elif stname is None or stname == "" or stname.isspace() == True:
                    if fulladdress[0].isdigit():
                        row[11] = (fulladdress.rsplit(' ')[-1].strip())
                    else:
                        row[11] = ""
                else:
                    print("Error")
            else:
                row[11] = ""
            # set postal town with spatial join
            # set postal state
            row[12] = "NV"
            # set postal zip with spatial join
            # set owner first name
            ownfirst = row[13]
            if not (ownfirst is None or ownfirst.isspace() == True):
                row[14] = ownfirst
            else:
                row[14] = ""
            # set owner last name
            ownlast = row[15]
            if not (ownlast is None or ownlast.isspace() == True):
                row[16] = ownlast
            else:
                row[16] = ""
            #set owner full name
            if not (ownfirst is None and ownlast is None):
                row[17] = (ownfirst + " " + ownlast).strip()
            else:
                row[17] = ""
            # set mail1
            mail1 = row[18]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True or mail1 == "NONE" or mail1 == "NOT SUPPLIED"):
                row[19] = mail1
            else:
                row[19] = ""
            # set mial address 2
            mail2 = row[20]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and ("ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[21] = mail2
            else:
                row[21] = ""
            # set mail city
            mailcity = row[22]
            if not (mailcity is None or mailcity == "" or mailcity == "NONE" or mailcity.isspace() == True):
                row[23] = mailcity
            else:
                row[23] = ""
            # set mail state
            mailstate = row[24]
            if not (mailstate is None or mailstate == "" or mailstate.isspace() == True):
                row[25] = mailstate
            else:
                row[25] = ""
            # set mail zip
            mailzip = row[26]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[27] = mailzip[:5]
            else:
                row[27] = ""
            # set jurisdiction
            row[28] = County_ABR
            # set county
            row[29] = County_ABR
            # set ownership type
            if not (ownlast is None or ownlast == "" or ownlast.isspace() == True):
                if ownlast in fedOwnList:
                    row[30] = "Federal"
                elif ownlast in localOwnList:
                    row[30] = "Local"
                elif ownlast in stateOwnList:
                    row[30] = "State"
                elif not ownlast in (fedOwnList, localOwnList, stateOwnList):
                    row[30] = "Private"
            # set county land use code
            ctyluc = row[31]
            if not (ctyluc is None or ctyluc == "" or ctyluc.isspace() == True):
                intctyluc = (ctyluc.rsplit(",", 1)[0].strip())
                row[32] = int(intctyluc)
            else:
                row[32] = ""
            # set assessed land value
            landval = row[33]
            row[34] = landval
            # set assessed improved value
            improval = row[35]
            row[36] = improval
            # set assessed sum
            if not (landval is None or improval is None):
                row[37] = landval + improval
            else:
                row[37] = ""
            # set tax land value
            taxlandval = row[38]
            if not taxlandval is None:
                row[39] = taxlandval
            else:
                row[39] = ""
            # set tax improved value
            taximproval = row[40]
            if not taximproval is None:
                row[41] = taximproval
            else:
                row[41] = ""
            # set tax sum
            if not (taxlandval is None or taximproval is None):
                row[42] = taxlandval + taximproval
            else:
                row[42] = ""
            # set tax year
            row[43] = current_year
            # set dwelling units
            dwellingunit = row[44]
            if not (dwellingunit is None or dwellingunit == ""):
                row[45] = dwellingunit
            else:
                row[45] = ""
            # set bedroom units
            bedunit = row[46]
            if not (bedunit is None or bedunit == ""):
                row[47] = bedunit
            else:
                row[47] = ""
            # set bathroom units
            bathunit = row[48]
            if not (bathunit is None or bathunit == ""):
                row[49] = bathunit
            else:
                row[49] = ""
            # set year built
            yrbuilt = row[50]
            if not (yrbuilt is None or yrbuilt == ""):
                row[51] = yrbuilt
            else:
                row[51] = ""
            # set building sqfeet
            sqfeet = row[52]
            if not (sqfeet is None or sqfeet == ""):
                row[53] = sqfeet
            else:
                row[53] = 0
            # update all rows
            cursor.updateRow(row)
        print ("Rows in {} county staging dataset have been updated".format(County_List[4]))        
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Carson":
    print ("Processing the {} dataset".format(County_List[0]))
    field_names = ['APN', 'APN_NEW', 'PPNO_NEW','Location_Number','HSE_NUMBR_NEW','Location_Unit___s_','UNIT_NUMBR_NEW', 
                   'Location_Direction', 'STR_DIR_NEW', 'Location_or_Street_Name', 'STR_NAME_NEW', 'STR_SUFFIX_NEW', 'PSTL_TOWN_NEW', 
                   'PSTL_STATE_NEW', 'PSTL_ZIP5_NEW', 'Legal_Owner', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'Mailing_Address_Line_1','MAIL_ADD1_NEW', 'Mailing_Address_Line_2', 'MAIL_ADD2_NEW', 'Mailing_City_State', 
                   'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'Mailing_Zip_Code',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW', 'COUNTY', 
                   'OWNERSHIP_TYPE', 'Land_Use_Code', 'COUNTY_LANDUSE_CODE', 'Land_Value', 'AS_LANDVALUE_NEW', 'Improvements_Value',
                   'AS_IMPROVALUE_NEW', 'AS_SUM_NEW', 'TAX_LANDVALUE_NEW', 'TAX_IMPROVALUE_NEW', 'TAX_SUM_NEW', 
                   'TAX_YEAR_NEW', 'Total___Dwelling_Units', 'UNITS_NEW', 'F__of_Beds', 'BEDROOMS_NEW', 'F__of_Baths', 'BATHROOMS_NEW',
                   'Original_Construction_Year', 'YEAR_BUILT_NEW', 'Buildings_Sq_Feet', 'BUILDING_SQFT_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set APN
            apn = row[0]
            row[1] = (apn[:3] + sep + apn[3:6] + sep + apn[6:8])
            # set PPNO
            row[2] = (apn)
            # set house number
            hsenum = row[3]
            if not (hsenum is None): 
                row[4] = hsenum
            else:
                row[4] = 0
            # set unit number
            unit = row[5]
            if not (unit is None):
                row[6] = "#" + unit
            else:
                row[6] = ""
            # set street direction
            stdir = row[7]
            if not (stdir is None or stdir == "" or stdir.isspace() == True):
                row[8] = stdir
            else:
                row[8] = ""
            # set street name 
            stname = row[9]
            if not (stname is None or stname == "" or stname.isspace() == True):
                row[10] = stname
            else:
                row[10] = ""
            # set street suffix
            row[11] = ""
            # APO Address set at the end
            # set postal town
            row[12] = ''
            # set postal state
            row[13] = 'NV'
            # set postal zip
            row[14] = '89705' 
            # set owner first name
            own = row[15]
            row[16] = ""
            # set owner last name
            if not (own is None or own == "" or own.isspace() == True):
                row[17] = own
            else:
                row[17] = ""
            # set owner full name
            if not (own is None or own == "" or own.isspace() == True):
                row[18] = own
            else:
                row[18] = ""
            # set mail address 1
            mail1 = row[19]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True):
                row[20] = mail1.replace("%", "").strip()
            else:
                row[20] = ""
            # set mail address 2
            mail2 = row[21]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and (
            "ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[22] = mail2
            else:
                row[22] = ""
            # set mail city
            mailcity = row[23]
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True):
                row[24] = mailcity.split(",")[0].strip()
            else:
                row[24] = ""
            # set mail state
            mailstate = row[23]
            if not (mailstate is None):
                if "," in mailstate:
                    mailstate = mailcity.split(",")[1].strip()
                    row[25] = mailstate
                else:
                    row[25] = ""
            else:
                row[25] = ""
            # set mail zip
            mailzip = row[26]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[27] = mailzip[:5]
            else:
                row[27] = ""
            # set jurisdiction
            row[28] = County_ABR
            #set county
            row[29] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[30] = "Federal"
                elif own in localOwnList:
                    row[30] = "Local"
                elif own in stateOwnList:
                    row[30] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[30] = "Private"
            # set county land use
            ctyluc = row[31]
            if ctyluc is not None:
                row[32] = ctyluc
            else:
                row[32] = ""
            # set assessed land value
            landval = row[33]
            if not (landval is None):
                row[34] = landval
            else:
                row[34] = 0
            # set assessed improved value
            improval = row[35]
            if not (improval is None):
                row[36] = improval
            else:
                row[36] = 0
            # set assessed sum
            if not (landval is None or improval is None):
                row[37] = landval + improval
            else:
                row[37] = 0
            # set tax land value
            if not (landval is None):
                taxland = landval / 0.35
                row[38] = taxland
            else:
                row[38] = 0
            # set tax improved value
            if not (improval is None):
                taximprov = improval / 0.35
                row[39] = taximprov
            else:
                row[39] = 0
            # set tax sum value
            if not (taxland is None or taximprov is None):
                row[40] = taxland + taximprov
            else:
                row[40] = 0
            # set tax year
            row[41] = current_year
            # set dwelling units
            dwellingunit = row[42]
            if not (dwellingunit is None):
                row[43] = dwellingunit
            else:
                row[43] = ""
            # set bedroom units
            bedunit = row[44]
            if not (bedunit is None):
                row[45] = bedunit
            else:
                row[45] = ""
            # set bathroom units
            bathunit = row[46]
            if not (bathunit is None):
                row[47] = bathunit
            else:
                row[47] = ""
            # set year built
            yrbuilt = row[48]
            if not (yrbuilt is None):
                row[49] = yrbuilt
            else:
                row[49] = ""
            # set building sqfeet
            sqfeet = row[50]
            if not (sqfeet is None):
                row[51] = sqfeet
            else:
                row[51] = 0
            # update all rows
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[0]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Douglas":
    print ("Processing the {} dataset".format(County_List[1]))
    field_names = ['PIN', 'APN_NEW', 'APN', 'PPNO_NEW','PLOC_','HSE_NUMBR_NEW','PLOCU_','UNIT_NUMBR_NEW', 'PLOCDR', 'STR_DIR_NEW',
                   'PLOCNM', 'STR_NAME_NEW', 'PLOCTP', 'STR_SUFFIX_NEW', 'PSTL_TOWN_NEW', 'PSTL_STATE_NEW', 'PSTL_ZIP5_NEW',
                   'PANAME', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW','PMADD1', 'MAIL_ADD1_NEW', 'PMADD2',
                   'MAIL_ADD2_NEW', 'PMCTST', 'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'PZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW',
                   'COUNTY', 'OWNERSHIP_TYPE', 'UNITS_NEW', 'YLDUSE', 'COUNTY_LANDUSE_CODE', 'YLANDV', 'AS_LANDVALUE_NEW',
                   'YIMPRV', 'AS_IMPROVALUE_NEW', 'AS_SUM_NEW', 'TAX_LANDVALUE_NEW', 'TAX_IMPROVALUE_NEW', 'TAX_SUM_NEW',
                   'TAX_YEAR_NEW', 'P_DWEL', 'UNITS_NEW', 'PBEDS', 'BEDROOMS_NEW', 'PBATHS', 'BATHROOMS_NEW',
                   'PCONYR', 'YEAR_BUILT_NEW', 'PRESSF', 'BUILDING_SQFT_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == ""):
                row[1] = apn
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not ppno is None:
                row[3] = ppno
            else:
                row[3] = 0
            # set house number
            hsenum = row[4]
            if not (hsenum is None):
                row[5] = hsenum
            else:
                row[5] = 0
            # set unit number
            unit = row[6]
            if not (unit is None or unit == "" or unit.isspace() == True):
                row[7] = "#" + unit
            else:
                row[7] = ""
            # set street direction
            stdir = row[8]
            if not stdir is None:
                row[9] = stdir
            else:
                row[9] = ""
            # set street name
            stname = row[10]
            if not stname is None:
                row[11] = stname
            else:
                row[11] = ""
            # set street suffix
            stsuf = row[12]
            if not stsuf is None:
                row[13] = stsuf
            else:
                row[13] = ""
            # set postal town
            row[14] = ""
            # set postal state
            row[15] = "NV"
            # set postal zip
            row[16] = ""
            # set owner fist name
            ownfirst = row[17]
            if not (ownfirst is None or ownfirst == "" or ownfirst.isspace() == True):
                try:
                    if ownfirst not in ownerListVal:
                        row[18] = ownfirst.split(",", 1)[1].strip()
                    else:
                        row[18] = ""
                except IndexError:
                    row[18] = ""
            else:
                row[18] = ""
            # set owner last name
            ownlast = row[17]
            if not (ownlast is None or ownlast == "" or ownlast.isspace() == True):
                try:
                    if ownlast in ownerListVal:
                        row[19] = ownlast
                    else:
                        row[19] = ownlast.split(",")[0].strip()
                except IndexError:
                    row[19] = ""
            else:
                row[19] = ""
            # set owner fullname
            own = row[17]
            if not (own is None):
                row[20] = own.strip()
            else:
                row[20] = ""
            # set mail address 1
            mail1 = row[23]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True):
                row[22] = mail1.strip()
            else:
                row[22] = ""
            # set mail address 2
            mail2 = row[21]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and ("ATTN" in mail2 or "C/O" in mail2 or "PO BOX" in mail2 or "PMB" in mail2):
                row[24] = mail2.strip()
            else:
                row[24] = ""
            # set mail city
            mailcity = row[25]
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True):
                if "," in mailcity:
                    row[26] = mailcity.split(",")[0].strip()
            else:
                row[26] = ""
            # set mail state
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True):
                if "," in mailcity:
                    mailstate = mailcity.split(",")[1].strip()
                    if len(mailstate) > 2:
                        row[27] = ""
                        print ("International Address, {}".format(mailstate))
                    else:
                        row[27] = mailstate
            else:
                row[27] = ""
            # set mail zip code
            mailzip = row[28]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[29] = mailzip[:5]
            else:
                row[29] = ""
            # set jurisdiction
            row[30] = County_ABR
            # set county
            row[31] = County_ABR
            # set ownership type
            own = row[17]
            if not (own is None or own == "" or own.isspace() == True):
                ownership = (own.strip())
                if ownership in fedOwnList:
                    row[32] = "Federal"
                elif ownership in localOwnList:
                    row[32] = "Local"
                elif ownership in stateOwnList:
                    row[32] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[32] = "Private"
            # set units
            row[33] = 0
            # set county land use code
            ctyluc = row[34]
            row[35] = ctyluc
            # set assessed land value
            landval = row[36]
            if not (landval is None):
                row[37] = landval
            else:
                row[37] = 0
            # set assessed improved value
            improval = row[38]
            if not (improval is None):
                row[39] = improval
            else:
                row[39] = 0
            # set assessed sum
            if not (landval is None or improval is None):
                row[40] = landval + improval
            else:
                row[40] = 0
            # set tax land value
            if not (landval is None):
                taxland = landval / 0.35
                row[41] = taxland
            # set tax improved value
            if not (improval is None):
                taximprov = improval / 0.35
                row[42] = taximprov
            # set tax sum value
            if not (taxland is None or taximprov is None):
                row[43] = taxland + taximprov
            # set tax year
            row[44] = current_year
            # set dwelling units
            dwellingunit = row[45]
            if not (dwellingunit is None):
                row[46] = dwellingunit
            else:
                row[46] = ""
            # set bedroom units
            bedunit = row[47]
            if not (bedunit is None):
                row[48] = bedunit
            else:
                row[48] = ""
            # set bathroom units
            bathunit = row[49]
            if not (bathunit is None):
                row[50] = bathunit
            else:
                row[50] = ""
            # set year built
            yrbuilt = row[51]
            if not (yrbuilt is None):
                row[52] = yrbuilt
            else:
                row[52] = ""
            # set building sqfeet
            sqfeet = row[53]
            if not (sqfeet is None):
                row[54] = sqfeet
            else:
                row[54] = 0
            # update all rows  
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[1]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "El_Dorado":
    print("Processing the {} dataset".format(County_List[2]))
    field_names = ['PRCL_ID', 'APN_NEW','PPNO_NEW','ADDRSTNBR','HSE_NUMBR_NEW','ADDRUNITNB','UNIT_NUMBR_NEW', 
                   'ADDRSTDIR', 'STR_DIR_NEW', 'ADDRSTPRFX', 'ADDRSTNAME', 'STR_NAME_NEW', 'ADDRSTTYPE', 'STR_SUFFIX_NEW', 'PSTL_STATE_NEW',
                   'OWNER_NAME', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW', 'MAIL_ADDR1', 'MAIL_ADDR2', 'MAIL_ADDR3', 
                   'MAIL_ADDR4', 'MAIL_ADD1_NEW', 'MAIL_ADD2_NEW', 'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'MAIL_ZIP5_NEW', 
                   'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE', 'USECD_1', 'COUNTY_LANDUSE_CODE', 'LAND_VAL', 
                   'AS_LANDVALUE_NEW', 'STRUCT_VAL', 'AS_IMPROVALUE_NEW', 'AS_SUM_NEW', 'TAX_LANDVALUE_NEW', 
                   'TAX_IMPROVALUE_NEW', 'TAX_SUM_NEW', 'TAX_YEAR_NEW', 'DWELLUNITS', 'UNITS_NEW', 'BEDROOMS', 
                   'BEDROOMS_NEW', 'BATHROOMS_NEW', 'YR_BUILT', 'YEAR_BUILT_NEW', 'IMPR_SQ_FT', 'BUILDING_SQFT_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn):
                row[1] = (apn[:3] + sep + apn[3:6] + sep + apn[6:9])
            else:
                row[1] = ""
            # set ppno
            ppno = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn or 'NP' in apn):
                try:
                    row[2] = float(ppno)
                except ValueError:
                    row[2] = 0
            else:
                row[2] = 0
             # set house number
            hsenum = row[3]
            row[4] = hsenum
            # set unit number
            unit = row[5]
            if not (unit is None or unit == "" or unit.isspace() == True):
                row[6] = "#" + unit
            else:
                row[6] = ""
            # set street direction
            stdir = row[7]
            if not (stdir is None or stdir == "" or stdir.isspace() == True or 'UNASSIGNED' in stdir):
                row[8] = stdir[:1]
            else:
                row[8] = ""
            # set street name
            stprefix = row[9]
            stname = row[10]
            if not (stname is None or stname == "" or stname.isspace() == True):
                if not (stprefix is None or stprefix == "" or stprefix.isspace() == True):
                    row[11] = (stprefix.strip()) + " " + (stname.strip())
                else:
                    row[11] = (stname.strip())
            else:
                row[11] = ""
            # set street suffix
            stsuff = row[12]
            if not (stsuff is None or stsuff == "" or stsuff.isspace() == True or 'UNASSIGNED' in stsuff):
                row[13] = stsuff
            else:
                row[13] = ""
            # set postal town and postal zip with spatial join....later in the python notebook....scroll down
            # set postal state
            row[14] = "CA"
            # set owner first name
            own = row[15]
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own not in ownerListVal:
                        row[16] = own.split(" ", 1)[1].strip()
                except IndexError:
                    row[16] = ""
            else:
                row[16] = ""
            # set owner last name
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own in ownerListVal:
                        row[17] = own
                    else:
                        row[17] = own.split(" ")[0].strip()
                except IndexError:
                    row[17] = ""
            # If 4 mail address fields exist use script from CSLT
            # If not use the following
            # set mail1
            #mail1 = row[19]
            #if not mail1 is None or mail1 == "" or mail1.isspace() == True:
                #row[23] = mail1.replace("%", "").strip()
            #else:
                #row[23] = ""
            # set mail2
            #row[24] = ""
            # set mail city
            #mailcity = row[20]
            #if not mailcity is None or mailcity == "" or mailcity.isspace() == True:
                #row[25] = mailcity
            #else:
                #row[25] = ""
            # set mail state
            #mailstate = row[21]
            #if not mailstate is None or mailstate == "" or mailstate.isspace() == True:
                #row[26] = mailstate
            #else:
                #row[26] = ""
            # set mail zip
            #mailzip = row[22]
            #if not mailzip is None or mailzip == "" or mailzip.isspace() == True:
                #row[27] = mailzip[:5]
            #else:
                #row[27] = ""
            # set jurisdiction
            row[28] = County_ABR
            # set county
            row[29] = County_ABR
            # set ownership type
            own = row[15]
            if not (own is None or own == "" or own.isspace() == True):
                ownership = (own.strip())
                if ownership in fedOwnList:
                    row[30] = "Federal"
                elif ownership in localOwnList:
                    row[30] = "Local"
                elif ownership in stateOwnList:
                    row[30] = "State"
                else:
                    row[30] = "Private"
            # set county land use code
            ctyluc = row[31]
            if not ctyluc is None or ctyluc == "" or ctyluc.isspace() == True:
                row[32] = ctyluc
            else:
                row[32] = ""
            # set assessed land value
            landval = row[33]
            row[34] = landval
            # set assessed improved value
            improval = row[35]
            row[36] = improval
            # set assessed sum
            row[37] = improval + landval
            # set tax land value
            taxland = row[33]
            row[38] = taxland
            # set tax improved value
            taximprov = row[35]
            row[39] = taximprov
            # set tax sum
            row[40] = taxland + taximprov
            # set tax year
            row[41] = current_year
            # set dwelling units
            dwellingunit = row[42]
            if not (dwellingunit is None):
                row[43] = dwellingunit
            else:
                row[43] = ""
            # set bedroom units
            bedunit = row[44]
            if not (bedunit is None):
                row[45] = bedunit
            else:
                row[45] = ""
            # set bathroom units
            row[46] = "N/A"
            # set year built
            yrbuilt = row[47]
            if not (yrbuilt is None):
                row[48] = yrbuilt
            else:
                row[48] = ""
            # set building sqfeet
            sqfeet = row[49]
            if not (sqfeet is None):
                row[50] = sqfeet
            else:
                row[50] = 0
            # update all rows 
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[2]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Placer":
    print ("Processing the {} dataset".format(County_List[3]))
    field_names = ['APN', 'APN_NEW','GISAPN', 'PPNO_NEW','STREETNUM_1','HSE_NUMBR_NEW','SP_APT','UNIT_NUMBR_NEW', 
                   'STREETDIR', 'STR_DIR_NEW','STREETNAME_1', 'STR_NAME_NEW', 'STREETTYPE_1', 'STR_SUFFIX_NEW', 'PSTL_STATE_NEW',
                   'OWNER1', 'OWN_FIRST_NEW', 'OWNER2', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'ADR1_1','MAIL_ADD1_NEW', 'MAIL_ADD2_NEW','CITY_1', 'MAIL_CITY_NEW', 'STATE_1','MAIL_STATE_NEW', 
                   'ZIP_1',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE',
                   'USE_CD', 'COUNTY_LANDUSE_CODE', 'USE_CD_N_1', 'COUNTY_LANDUSE_DESCRIPTION',
                   'LANDVALUE_1', 'AS_LANDVALUE_NEW','STRUCTURE_1','AS_IMPROVALUE_NEW','AS_SUM_NEW',
                   'TAX_LANDVALUE_NEW','TAX_IMPROVALUE_NEW','TAX_SUM_NEW','TAX_YEAR_NEW','UNITS_NEW', 'BEDROOMS_NEW', 
                   'BATHROOMS_NEW', 'YEAR_BUILT_NEW', 'BUILDING_SQFT_NEW']
   # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or "ROW" in apn or len(apn) < 8):
                row[1] = apn[:11]
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not (ppno is None or ppno == "" or ppno.isspace() == True or "ROW" in ppno or len(ppno) < 8):
                row[3] = int(ppno)
            else:
                row[3] = 0
            # set house number
            hsenumbr = row[3]
            if not (hsenumbr is None or hsenumbr == ""):
                row[4] = hsenumbr
            else:
                row[4] = 0
            # set unit number
            unit = row[6]
            if not (unit is None):
                row[7] = "#" + unit
            else:
                row[7] = ""
            # set street direction
            stdir = row[8]
            if not stdir is None:
                row[9] = stdir
            else:
                row[9] = ""
            # set street name
            stname = row[10]
            if not stname is None:
                row[11] = stname
            else:
                row[11] = ""
            # set street suffix
            stsuff = row[12]
            if not stsuff is None:
                row[13] = stsuff
            else:
                row[13] = ""
            # set postal state
            row[14] =  "CA"
            # set postal town and postal zip with spatial join....scroll down
            #set owner full name
            own = row[15]
            if not (own is None or own == "" or own.isspace() == True):
                row[19] = own
            else:
                row[19] = ""
            #set owner first name
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own not in ownerListVal:
                        row[16] = own.split(" ", 1)[1].strip()
                except IndexError:
                    row[16] = ""
            else:
                row[16] = ""
            #set owner last name
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own in ownerListVal:
                        row[18] = own
                    else:
                        row[18] = own.split(" ")[0].strip()
                except IndexError:
                    row[18] = ""
            else:
                row[18] = ""
            # set mail1
            mail1 = row[20]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True):
                row[21] = mail1
            else:
                row[21] = ""
            # set mail2
            mail2 = row[20]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and ("ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[22] = mail2
            else:
                row[22] = ""
            # set mail city
            mailcity = row[23]
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True or "N/A" in mailcity or \
                    "N\A" in mailcity or "P O BOX" in mailcity or "VALUE" in mailcity or "PRORATED" in mailcity):
                row[24] = mailcity
            else:
                row[24] = ""
            # set mail state
            mailstate = row[25]
            if not (mailstate is None or mailstate == "" or mailstate.isspace() == True or len(mailstate) < 2):
                row[26] = mailstate
            else:
                row[26] = ""
            # set mail zip
            mailzip = row[27]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[28] = mailzip[:5]
            else:
                row[28] = ""
            # set jurisdiction
            row[29] = County_ABR
            # set county
            row[30] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[31] = "Federal"
                elif own in localOwnList:
                    row[31] = "Local"
                elif own in stateOwnList:
                    row[31] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[31] = "Private"
            # set county land use code
            ctyluc = row[32]
            if not (ctyluc is None or ctyluc == "" or ctyluc.isspace() == True):
                row[33] = ctyluc
            else:
                row[33] = ""
            # set assessed land value
            landval = row[36]
            row[37] = landval
            # set assessed improved value
            improval = row[38]
            row[39] = improval
            # set assessed sum
#             if not (landval is None and improval is None):
#                 row[40] =  landval + improval
#             else:
#                 row[40] = ""
            # set tax land value
#             if not (landval is None):
#                 row[41] =  landval
#             else:
#                 row[41] = ""
            # set tax impoved value
#             if not (improval is None):
#                 row[42] =  improval
#             else:
#                 row[42] = ""
            # set tax sum
#             if not (landval is None and improval is None):
#                 row[43] =  landval + improval
#             else:
#                 row[43] = ""
            # set tax year
            row[44] = current_year
            # set dwelling units
            #dwellingunit = row[41]
            #if not (dwellingunit is None):
                #row[42] = dwellingunit
            #else:
                #row[42] = ""
            # set bedroom units
            #bedunit = row[43]
            #if not (bedunit is None):
                #row[44] = bedunit
            #else:
                #row[44] = ""
            # set bathroom units
            #row[45] = ""
            # set year built
            #yrbuilt = row[46]
            #if not (yrbuilt is None):
                #row[47] = yrbuilt
            #else:
                #row[47] = ""
            # set building sqfeet
            #52&53
#             sqfeet = row[32]
#             if not (sqfeet is None):
#                 row[33] = sqfeet
#             else:
#                 row[33] = 0
            
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[3]))
#----------------------------------------------------------------------------------------            
# Calculate APO_ADDRESS
expression = "concat(!HSE_NUMBR_NEW!, !STR_DIR_NEW!, !STR_NAME_NEW!, !STR_SUFFIX_NEW!, !UNIT_NUMBR_NEW!)"
apo_codeblock = """
def concat(*args):
    retval = ""
    sep = " "
    for t in args:
        s = str(t).strip()
        if s != None:
            retval += sep + s
    return retval.lstrip(sep)"""
arcpy.CalculateField_management(parcel_out_data, "APO_ADDRESS_NEW", expression, python_version, apo_codeblock)
print ("Calculated APO_ADDRESS")
# arcpy.CalculateField_management(parcel_out_data, "PARCEL_ACRES_NEW", "!shape.area@acres!", python_version, "")
# print ("Calculated Parcel Acreage")
# arcpy.CalculateField_management(parcel_out_data, "PARCEL_SQFT_NEW", "!shape.area@squarefeet!", python_version, "")
# print ("Calculated Parcel SQFT")    
arcpy.CalculateField_management(parcel_out_data, "APO_ADDRESS_NEW", "' '.join(!APO_ADDRESS_NEW!.strip().split())", python_version,"#")
print ("Removed Double Spaces from APO_ADDRESS")

#---------------------------------------------------------------------------------------- 
# Calculate PARCEL_ACRES, and PARCEL_SQFT
print("Calculating Acres...")
with arcpy.da.UpdateCursor(parcel_out_data, ['PARCEL_ACRES_NEW', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'ACRES')
        cursor.updateRow(row)

print("Calculating Square Feet...")
with arcpy.da.UpdateCursor(parcel_out_data, ['PARCEL_SQFT_NEW', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'FEET')
        cursor.updateRow(row)

print("Complete")

Processing the El_Dorado dataset
0 PRCL_ID
1 APN_NEW
2 PPNO_NEW
3 ADDRSTNBR
4 HSE_NUMBR_NEW
5 ADDRUNITNB
6 UNIT_NUMBR_NEW
7 ADDRSTDIR
8 STR_DIR_NEW
9 ADDRSTPRFX
10 ADDRSTNAME
11 STR_NAME_NEW
12 ADDRSTTYPE
13 STR_SUFFIX_NEW
14 PSTL_STATE_NEW
15 OWNER_NAME
16 OWN_FIRST_NEW
17 OWN_LAST_NEW
18 OWN_FULL_NEW
19 MAIL_ADDR1
20 MAIL_ADDR2
21 MAIL_ADDR3
22 MAIL_ADDR4
23 MAIL_ADD1_NEW
24 MAIL_ADD2_NEW
25 MAIL_CITY_NEW
26 MAIL_STATE_NEW
27 MAIL_ZIP5_NEW
28 JURISDICTION_NEW
29 COUNTY
30 OWNERSHIP_TYPE
31 USECD_1
32 COUNTY_LANDUSE_CODE
33 LAND_VAL
34 AS_LANDVALUE_NEW
35 STRUCT_VAL
36 AS_IMPROVALUE_NEW
37 AS_SUM_NEW
38 TAX_LANDVALUE_NEW
39 TAX_IMPROVALUE_NEW
40 TAX_SUM_NEW
41 TAX_YEAR_NEW
42 DWELLUNITS
43 UNITS_NEW
44 BEDROOMS
45 BEDROOMS_NEW
46 BATHROOMS_NEW
47 YR_BUILT
48 YEAR_BUILT_NEW
49 IMPR_SQ_FT
50 BUILDING_SQFT_NEW
Rows in El_Dorado dataset have been updated
Calculated APO_ADDRESS
Removed Double Spaces from APO_ADDRESS
Calculating Acres...
Calculating Square Feet...
Complete


### Merge Staging Datasets and Eliminate County Fields

In [2]:
# Parcel staging feature classes to be merged
staging = "F:\\GIS\\ParcelUpdate\\" + w_folder + ""
WA_Staging = staging + "\\Washoe\\Modified\\WA_Staging.gdb\\Parcels\\Washoe_Parcels"
CC_Staging = staging + "\\Carson\\Modified\\CC_Staging.gdb\\Parcels\\Carson_Parcels"
DG_Staging = staging + "\\Douglas\\Modified\\DG_Staging.gdb\\Parcels\\Douglas_Parcels"
EL_Staging = staging + "\\El_Dorado\Modified\EL_Staging.gdb\Parcels\\El_Dorado_Parcels"
PL_Staging = staging + "\\Placer\\Modified\\PL_Staging.gdb\\Parcels\\Placer_Parcels"

# set output staging feature class
Parcel_Master = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_New"

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()

# Add all fields
fieldMappings.addTable(WA_Staging)
fieldMappings.addTable(CC_Staging)
fieldMappings.addTable(DG_Staging)
fieldMappings.addTable(EL_Staging)
fieldMappings.addTable(PL_Staging)

# Remove all output fields from the field mappings, except fields in field_master list
for field in fieldMappings.fields:
    if field.name not in ['APN_NEW', 'PPNO_NEW', 'HSE_NUMBR_NEW', 'UNIT_NUMBR_NEW', 'STR_DIR_NEW', 'STR_NAME_NEW', 
                'STR_SUFFIX_NEW', 'APO_ADDRESS_NEW', 'PSTL_TOWN_NEW', 'PSTL_STATE_NEW', 'PSTL_ZIP5_NEW', 
                'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW', 'MAIL_ADD1_NEW', 'MAIL_ADD2_NEW', 
                'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'MAIL_ZIP5_NEW', 'JURISDICTION_NEW', 'COUNTY', 'OWNERSHIP_TYPE', 
                'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
                'REGIONAL_LANDUSE', 'UNITS_NEW', 'YEAR_BUILT', 'BEDROOMS_NEW', 'BATHROOMS_NEW', 'BUILDING_SQFT',
                'ALLOWABLE_COVERAGE_BAILEY_SQFT', 'IMPERVIOUS_SURFACE_SQFT', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 
                'WATERSHED_NUMBER', 'WATERSHED_NAME', 'PRIORITY_WATERSHED', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 
                'AS_LANDVALUE_NEW', 'AS_IMPROVALUE_NEW', 'AS_SUM_NEW', 'TAX_LANDVALUE_NEW', 'TAX_IMPROVALUE_NEW', 
                'TAX_SUM_NEW', 'TAX_YEAR_NEW', 'PLAN_ID', 'PLAN_NAME', 'ZONING_ID', 'ZONING_DESCRIPTION', 'TOLERANCE_ID', 
                'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'INDEX_1987', 'LOCAL_PLAN_HYPERLINK', 
                'DESIGN_GUIDELINES_HYPERLINK', 'LTINFO_HYPERLINK', 'INDEX_1987_HYPERLINK', 'TAZ',
                'PARCEL_ACRES_NEW', 'PARCEL_SQFT_NEW', 'DUPLICATE']:
        fieldMappings.removeFieldMap(fieldMappings.findFieldMapIndex(field.name))
for field in fieldMappings.fields:
    print ("Created Field Map:", field.name)

# Use Merge tool to move features into single dataset
arcpy.Merge_management([WA_Staging, CC_Staging, DG_Staging, EL_Staging, PL_Staging], Parcel_Master, fieldMappings)
print ("Created a new parcel dataset")

Created Field Map: APN_NEW
Created Field Map: PPNO_NEW
Created Field Map: HSE_NUMBR_NEW
Created Field Map: UNIT_NUMBR_NEW
Created Field Map: STR_DIR_NEW
Created Field Map: STR_NAME_NEW
Created Field Map: STR_SUFFIX_NEW
Created Field Map: APO_ADDRESS_NEW
Created Field Map: PSTL_TOWN_NEW
Created Field Map: PSTL_STATE_NEW
Created Field Map: PSTL_ZIP5_NEW
Created Field Map: OWN_FIRST_NEW
Created Field Map: OWN_LAST_NEW
Created Field Map: OWN_FULL_NEW
Created Field Map: MAIL_ADD1_NEW
Created Field Map: MAIL_ADD2_NEW
Created Field Map: MAIL_CITY_NEW
Created Field Map: MAIL_STATE_NEW
Created Field Map: MAIL_ZIP5_NEW
Created Field Map: JURISDICTION_NEW
Created Field Map: COUNTY
Created Field Map: OWNERSHIP_TYPE
Created Field Map: COUNTY_LANDUSE_CODE
Created Field Map: COUNTY_LANDUSE_DESCRIPTION
Created Field Map: TRPA_LANDUSE_DESCRIPTION
Created Field Map: REGIONAL_LANDUSE
Created Field Map: UNITS_NEW
Created Field Map: BEDROOMS_NEW
Created Field Map: BATHROOMS_NEW
Created Field Map: ALLOWABLE

### Change Field Names (eliminate the '_NEW')

In [3]:
import os
import arcpy

fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_New" 
new_fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master" 

field_mappings = arcpy.FieldMappings() # Create new field mapping object

#Loop through fields. For each field, create a corresponding FieldMap object.
#If the name ends with "_NEW", strip it off.
#Note: "FID and "Shape" are skipped

for field in arcpy.ListFields(fc):
    if not field.name == "OBJECTID" and not field.name == "Shape":
        old_name = field.name

        #Rename if necessary
        if old_name.endswith("_NEW"):
            new_name = old_name[:-4]
        else:
            new_name = old_name

        #Create new FieldMap object    
        new_f = arcpy.FieldMap()
        new_f.addInputField(fc, old_name) # Specify the input field to use

        #Rename output field
        new_f_name = new_f.outputField
        new_f_name.name = new_name
        new_f_name.aliasName = new_name
        new_f.outputField = new_f_name

        #Add field to FieldMappings object
        field_mappings.addFieldMap(new_f)

#Convert table using your created Field Mappings object
arcpy.FeatureClassToFeatureClass_conversion(fc, os.path.dirname(new_fc), os.path.basename(new_fc), field_mapping=field_mappings)

<Result 'F:\\GIS\\ParcelUpdate\\2021_04\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master'>

## Update Field Values

### Field Calculator Function

In [2]:
import arcpy

def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    
    #updateFC = r"C:\Path\UpdateFeatureClass"  
    #updateFieldsList = ["JoinField", "ValueField"]
    
    #sourceFC = r"C:\Path\SourceFeatureClass"  
    #sourceFieldsList = ["JoinField", "ValueField"]  
    
    # Use list comprehension to build a dictionary from arcpy SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

### Set local variables again...

In [3]:
import arcpy
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
ParcelPoint = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Point"

if arcpy.Exists(ParcelPoint):
    arcpy.Delete_management(ParcelPoint)
arcpy.FeatureToPoint_management(ParcelLayer, ParcelPoint, "INSIDE")

# File paths
sdeBase = "F:\\GIS\\DB_CONNECT\\Vector.sde"
#sde feature classes
sde_FireDistrict = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_NRCSSoils1974 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_HydroArea = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed = sdeBase + "\\sde.SDE.Water\\sde.SDE.Priority"
sde_RegionalLandUse = sdeBase + "\\sde.SDE.Planning\\sde.SDE.RegionalLandUse"
sde_LocalPlan = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_Zoning =  sdeBase + "\\sde.SDE.Planning\\sde.SDE.Zoning_LocalPlan"
sde_SpecialDistrict = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_TownCenter = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987 = sdeBase + "\\sde.SDE.Index\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_UrbanArea = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_Littoral = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_CSLT = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"
sde_CurrentParcels = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
sde_NewZoning = sdeBase + "\\sde.SDE.Planning\\sde.SDE.District"
sde_TAZ = sdeBase + "\\sde.SDE.Transportation\\sde.SDE.Transportation_Analysis_Zone"
sde_Tolerance = sdeBase + "\\sde.SDE.Shorezone\\sde.SDE.Tolerance_District"

# in memory files
wk_memory = "in_memory" + "\\"
ParcelPoint_FireDistrict = wk_memory + "ParcelPoint_FireDistrict"
ParcelPoint_Soils74 = wk_memory + "\\ParcelPoint_Soils74"
ParcelPoint_Soils03 = wk_memory + "\\ParcelPoint_Soils03"
ParcelPoint_HydroArea = wk_memory + "\\ParcelPoint_HydroArea"
ParcelPoint_Watershed = wk_memory + "\\ParcelPoint_Watershed"
ParcelPoint_RegionalLandUse = wk_memory + "\\ParcelPoint_RegionalLandUse"
ParcelPoint_LocalPlan = wk_memory + "\\ParcelPoint_LocalPlan"
ParcelPoint_TownCen6ter = wk_memory + "\\ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer = wk_memory + "\\ParcelPoint_TownCenterBuffer"
ParcelPoint_Zoning = wk_memory + "\\ParcelPoint_Zoning"
ParcelPoint_NewZoning = wk_memory + "\\ParcelPoint_NewZoning"
ParcelPoint_SpecialDistrict = wk_memory + "\\ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987 = wk_memory + "\\ParcelPoint_Index1987"
ParcelPoint_PstlTown = wk_memory + "\\ParcelPoint_PstlTown"
ParcelPoint_PstlZip = wk_memory + "\\ParcelPoint_PstlZip"
ParcelPoint_TRPAboundary= wk_memory + "\\ParcelPoint_TRPABoundary"
ParcelPoint_CSLT = wk_memory + "\\ParcelPoint_CSLT"
ParcelPoint_TAZ = wk_memory + "\\ParcelPoint_TAZ"
ParcelPoint_Design = wk_memory + "\\ParcelPoint_Design"
ParcelPoint_Tolerance = wk_memory + "\\ParcelPoint_Tolerance"
ParcelPoint_TAZ = wk_memory + "\\ParcelPoint_TAZ"

### Update Fire Protection District

In [4]:
# delete feature class if it exists already
if arcpy.Exists(ParcelPoint_FireDistrict):
    arcpy.Delete_management(ParcelPoint_FireDistrict)

print("Starting the Fire District Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_FireDistrict, ParcelPoint_FireDistrict, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Fire District Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'FIREPD'], ParcelPoint_FireDistrict, ['APN', 'DISTRICT'])
print ("The 'FIREPD' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["FIREPD"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Fire District Spatial Join
Finished the Fire District Spatial Join
Started data transfer: 2021-05-05 06:26:14
Finished data transfer: 2021-05-05 06:26:27
The 'FIREPD' field in the parcel data has been updated
NULL values have been updated


### Update Soils 1974 Field

In [5]:
# delete feature class if it exists already
if arcpy.Exists(ParcelPoint_Soils74):
    arcpy.Delete_management(ParcelPoint_Soils74)

print("Starting the SOIL_1974 Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils1974, ParcelPoint_Soils74, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_1974 Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SOIL_1974'], ParcelPoint_Soils74, ['APN', 'MUSYM_74'])
print ("The 'SOIL_1974' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["SOIL_1974"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the SOIL_1974 Spatial Join
Finished the SOIL_1974 Spatial Join
Started data transfer: 2021-05-05 06:27:14
Finished data transfer: 2021-05-05 06:27:22
The 'SOIL_1974' field in the parcel data has been updated
NULL values have been updated


### Update Soils 2003 Field

In [6]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Soils03):
    arcpy.Delete_management(ParcelPoint_Soils03)

print("Starting the SOIL_2003 Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils2003, ParcelPoint_Soils03, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_2003 Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SOIL_2003'], ParcelPoint_Soils03, ['APN', 'MUSYM_03'])
print ("The 'SOIL_2003' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["SOIL_2003"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the SOIL_2003 Spatial Join
Finished the SOIL_2003 Spatial Join
Started data transfer: 2021-05-05 06:28:52
Finished data transfer: 2021-05-05 06:29:02
The 'SOIL_2003' field in the parcel data has been updated
NULL values have been updated


### Update Hydrologic Area Field

In [7]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_HydroArea):
    arcpy.Delete_management(ParcelPoint_HydroArea)

print("Starting the Hyrdrologic Area Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_HydroArea, ParcelPoint_HydroArea, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Hydrologic Area Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'HRA_NAME'], ParcelPoint_HydroArea, ['APN', 'HRA_NAME_1'])
print ("The 'HRA_NAME' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["HRA_NAME"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Hyrdrologic Area Spatial Join
Finished the Hydrologic Area Spatial Join
Started data transfer: 2021-05-05 06:30:20
Finished data transfer: 2021-05-05 06:30:29
The 'HRA_NAME' field in the parcel data has been updated
NULL values have been updated


### Update Watershed Fields

In [8]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Watershed):
    arcpy.Delete_management(ParcelPoint_Watershed)

print("Starting the Watershed Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Watershed, ParcelPoint_Watershed, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Watershed Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'WATERSHED_NUMBER'], ParcelPoint_Watershed, ['APN', 'NUMBER'])
print ("The 'WATERSHED_NUMBER' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'WATERSHED_NAME'], ParcelPoint_Watershed, ['APN', 'NAME'])
print ("The 'WATERSHED_NAME' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'PRIORITY_WATERSHED'], ParcelPoint_Watershed, ['APN', 'PRIORITY'])
print ("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["WATERSHED_NUMBER"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = 0
            cursor.updateRow(row)
print ("WATERSHED_NUMBER NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["WATERSHED_NAME"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("WATERSHED_NAME NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["PRIORITY_WATERSHED"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = 0
            cursor.updateRow(row)
print ("PRIORITY_WATERSHED NULL values have been updated")

Starting the Watershed Spatial Join
Finished the Watershed Spatial Join
Started data transfer: 2021-05-05 06:31:45
Finished data transfer: 2021-05-05 06:31:55
The 'WATERSHED_NUMBER' field in the parcel data has been updated
Started data transfer: 2021-05-05 06:31:55
Finished data transfer: 2021-05-05 06:32:08
The 'WATERSHED_NAME' field in the parcel data has been updated
Started data transfer: 2021-05-05 06:32:08
Finished data transfer: 2021-05-05 06:32:18
The 'PRIORITY_WATERSHED' field in the parcel data has been updated
WATERSHED_NUMBER NULL values have been updated
WATERSHED_NAME NULL values have been updated
PRIORITY_WATERSHED NULL values have been updated


### Update Regional Land Use Field

In [9]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_RegionalLandUse):
    arcpy.Delete_management(ParcelPoint_RegionalLandUse)

print("Starting the Regional Land Use Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_RegionalLandUse, ParcelPoint_RegionalLandUse, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Regional Land Use Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'REGIONAL_LANDUSE'], ParcelPoint_RegionalLandUse, ['APN', 'Description'])
print ("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["REGIONAL_LANDUSE"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Regional Land Use Spatial Join
Finished the Regional Land Use Spatial Join
Started data transfer: 2021-05-05 06:40:27
Finished data transfer: 2021-05-05 06:40:40
The 'REGIONAL_LANDUSE' field in the parcel data has been updated
NULL values have been updated


### Update Local Plan Fields

In [10]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_LocalPlan):
    arcpy.Delete_management(ParcelPoint_LocalPlan)

print("Starting the Local Plan Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_LocalPlan, ParcelPoint_LocalPlan, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Local Plan Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PLAN_ID'], ParcelPoint_LocalPlan, ['APN', 'PLAN_ID_1'])
print ("The 'PAS_ID' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'PLAN_NAME'], ParcelPoint_LocalPlan, ['APN', 'PLAN_NAME_1'])
print ("The 'PAS_NAME' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'LOCAL_PLAN_HYPERLINK'], ParcelPoint_LocalPlan, ['APN', 'File_URL'])
print ("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["PLAN_ID"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("PLAN_ID NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["PLAN_NAME"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("PLAN_NAME NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["LOCAL_PLAN_HYPERLINK"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("LOCAL_PLAN_HYPERLINK NULL values have been updated")

Starting the Local Plan Spatial Join
Finished the Local Plan Spatial Join
Started data transfer: 2021-05-05 06:41:22
Finished data transfer: 2021-05-05 06:41:28
The 'PAS_ID' field in the parcel data has been updated
Started data transfer: 2021-05-05 06:41:28
Finished data transfer: 2021-05-05 06:41:35
The 'PAS_NAME' field in the parcel data has been updated
Started data transfer: 2021-05-05 06:41:35
Finished data transfer: 2021-05-05 06:41:45
The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated
PLAN_ID NULL values have been updated
PLAN_NAME NULL values have been updated
LOCAL_PLAN_HYPERLINK NULL values have been updated


### Update Town Center Field

In [11]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TownCenter):
    arcpy.Delete_management(ParcelPoint_TownCenter)

print("Starting the Town Center Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenter, ParcelPoint_TownCenter, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'TOWN_CENTER'], ParcelPoint_TownCenter, ['APN', 'NAME'])
print ("The 'TOWN_CENTER' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["TOWN_CENTER"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Town Center Spatial Join
Finished the Town Center Spatial Join
Started data transfer: 2021-05-05 06:42:34
Finished data transfer: 2021-05-05 06:42:40
The 'TOWN_CENTER' field in the parcel data has been updated
NULL values have been updated


### Update Town Center Buffer Field

In [12]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TownCenterBuffer):
    arcpy.Delete_management(ParcelPoint_TownCenterBuffer)

print("Starting the Town Center Buffer Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenterBuffer, ParcelPoint_TownCenterBuffer, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Buffer Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'LOCATION_TO_TOWNCENTER'], ParcelPoint_TownCenterBuffer, ['APN', 'BUFFER_NAME'])
print ("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["LOCATION_TO_TOWNCENTER"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Town Center Buffer Spatial Join
Finished the Town Center Buffer Spatial Join
Started data transfer: 2021-05-05 06:43:22
Finished data transfer: 2021-05-05 06:43:33
The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated
NULL values have been updated


### Update Zoning Fields

#### Update Design Guidelines

In [19]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Design):
    arcpy.Delete_management(ParcelPoint_Design)

print("Starting the Zoning Spatial Join")
# Process Zoning Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NewZoning, ParcelPoint_Design, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Zoning Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'DESIGN_GUIDELINES_HYPERLINK'], ParcelPoint_Design, ['APN', 'DESIGN_GUIDELINES_HYPERLINK_1'])
print ("The 'Design Guidelines URL' field in the parcel data has been updated")

# # change NULL values to blank values
# with arcpy.da.UpdateCursor(ParcelLayer, ["DESIGN_GUIDELINES_HYPERLINK"]) as cursor:
#     for row in cursor:
#         if row[0] == None:
#             row[0] = ""
#             cursor.updateRow(row)
# print ("NULL values have been updated")

Starting the Zoning Spatial Join
Finished the Zoning Spatial Join
Started data transfer: 2021-05-05 07:23:40
Finished data transfer: 2021-05-05 07:23:51
The 'Design Guidelines URL' field in the parcel data has been updated


#### Update Zoning IDs

In [14]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_NewZoning):
    arcpy.Delete_management(ParcelPoint_NewZoning)

print("Starting the Zoning Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NewZoning, ParcelPoint_NewZoning, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Zoning Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'ZONING_ID'], ParcelPoint_NewZoning, ['APN', 'ZONING_ID_1'])
print ("The Zoning ID field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'ZONING_DESCRIPTION'], ParcelPoint_NewZoning, ['APN', 'ZONING_DESCRIPTION_1'])
print ("The Zoning Description field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["ZONING_ID"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("ZONING_ID NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["ZONING_DESCRIPTION"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("ZONING_DESCRIPTION NULL values have been updated")

Starting the Zoning Spatial Join
Finished the Zoning Spatial Join
Started data transfer: 2021-05-05 07:01:26
Finished data transfer: 2021-05-05 07:01:33
The Zoning ID field in the parcel data has been updated
Started data transfer: 2021-05-05 07:01:33
Finished data transfer: 2021-05-05 07:01:40
The Zoning Description field in the parcel data has been updated
ZONING_ID NULL values have been updated
ZONING_DESCRIPTION NULL values have been updated


#### Update Tolerance ID

In [15]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Tolerance):
    arcpy.Delete_management(ParcelPoint_Tolerance)

print("Starting the Zoning Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Tolerance, ParcelPoint_Tolerance, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "INTERSECT", "", "")
print ("Finished the Zoning Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'TOLERANCE_ID'], ParcelPoint_Tolerance, ['APN', 'DISTRICT'])
print ("The Tolerance ID field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["TOLERANCE_ID"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Zoning Spatial Join
Finished the Zoning Spatial Join
Started data transfer: 2021-05-05 07:04:19
Finished data transfer: 2021-05-05 07:04:28
The Tolerance ID field in the parcel data has been updated
NULL values have been updated


### Update 1987 Index Fields

In [16]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Index1987):
    arcpy.Delete_management(ParcelPoint_Index1987)

print("Starting the 1987 Index Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Index1987, ParcelPoint_Index1987, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the 1987 Index Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'INDEX_1987'], ParcelPoint_Index1987, ['APN', 'MAP_NUMBER'])
print ("The 'INDEX_1987' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'INDEX_1987_HYPERLINK'], ParcelPoint_Index1987, ['APN', 'MAP_PATH'])
print ("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["INDEX_1987"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("INDEX_1987 NULL values have been updated")
with arcpy.da.UpdateCursor(ParcelLayer, ["INDEX_1987_HYPERLINK"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("INDEX_1987_HYPERLINK NULL values have been updated")

Starting the 1987 Index Spatial Join
Finished the 1987 Index Spatial Join
Started data transfer: 2021-05-05 07:07:03
Finished data transfer: 2021-05-05 07:07:15
The 'INDEX_1987' field in the parcel data has been updated
Started data transfer: 2021-05-05 07:07:15
Finished data transfer: 2021-05-05 07:07:29
The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated
INDEX_1987 NULL values have been updated
INDEX_1987_HYPERLINK NULL values have been updated


### Update Postal Town Field

In [7]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_PstlTown):
    arcpy.Delete_management(ParcelPoint_PstlTown)

print("Starting the Postal Town Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_UrbanArea, ParcelPoint_PstlTown, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Town Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PSTL_TOWN'], ParcelPoint_PstlTown, ['APN', 'Name'])
print ("The 'PSTL_TOWN' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["PSTL_TOWN"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Postal Town Spatial Join
Finished the Postal Town Spatial Join
Started data transfer: 2021-04-29 15:56:21
Finished data transfer: 2021-04-29 15:56:33
The 'PSTL_TOWN' field in the parcel data has been updated
NULL values have been updated


### Update Postal Zip Field

In [8]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_PstlZip):
    arcpy.Delete_management(ParcelPoint_PstlZip)

print("Starting the Postal Zip Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zip, ParcelPoint_PstlZip, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Zip Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PSTL_ZIP5'], ParcelPoint_PstlZip, ['APN', 'GEOID10'])
print ("The 'PSTL_ZIP5' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["PSTL_ZIP5"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = ""
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the Postal Zip Spatial Join
Finished the Postal Zip Spatial Join
Started data transfer: 2021-04-29 15:57:10
Finished data transfer: 2021-04-29 15:57:21
The 'PSTL_ZIP5' field in the parcel data has been updated
NULL values have been updated


### Add 'CSLT' to Jurisdiction field

In [6]:
csltParcels = arcpy.SelectLayerByLocation_management(ParcelLayer, "HAVE_THEIR_CENTER_IN", sde_CSLT, 0,   
                                                     "NEW_SELECTION")

with arcpy.da.UpdateCursor(csltParcels, ["JURISDICTION"]) as cursor:
    for row in cursor:
        row[0] = "CSLT"
        # update all rows
        cursor.updateRow(row)


### Update Within TRPA Boundary Field

In [21]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TRPAboundary):
    arcpy.Delete_management(ParcelPoint_TRPAboundary)

print("Starting the TRPA Boundary Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TRPAboundary, ParcelPoint_TRPAboundary, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the TRPA Boundary Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'WITHIN_TRPA_BNDY'], ParcelPoint_TRPAboundary, ['APN', 'WITHIN_TRPA_BNDY'])
print ("The 'WITHIN_TRPA_BNDY' field in the parcel data has been updated")

Starting the TRPA Boundary Spatial Join
Finished the TRPA Boundary Spatial Join
Started data transfer: 2020-10-21 18:08:21
Finished data transfer: 2020-10-21 18:08:27
The 'WITHIN_TRPA_BNDY' field in the parcel data has been updated


In [5]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [17]:
# Select all new parcels that have their center within old Littoral parcels
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           sde_TRPAboundary, 
                                                           0, 
                                                          'NEW_SELECTION')

# expose Littoral field
fields = ['WITHIN_TRPA_BNDY']

# Create update cursor for littoral
with arcpy.da.UpdateCursor(parcelSelect, fields) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
        
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

# Create update cursor for non-littoral
with arcpy.da.UpdateCursor(parcelSelect, fields) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)


### Update Littoral Field

#### Note
* Manually check this field. Script accounts for most littoral parcels but not all. 

In [18]:
# select old littoral parcels from old Parcel Master
oldLittoral = arcpy.SelectLayerByAttribute_management(sde_CurrentParcels, 
                                                          'NEW_SELECTION', 
                                                          '"LITTORAL" = 1')


# Select all new parcels that have their center within old Littoral parcels
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           oldLittoral, 
                                                           0, 
                                                          'NEW_SELECTION')

# expose Littoral field
fields = ['LITTORAL']

# Create update cursor for littoral
with arcpy.da.UpdateCursor(parcelSelect, fields) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
        
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

# Create update cursor for non-littoral
with arcpy.da.UpdateCursor(parcelSelect, fields) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)


### Update Allowed Coverage Field (based on Bailey)

In [3]:
import arcpy
arcpy.env.overwriteOutput = True

# SDE file path 
sdeBase = "F:\\GIS\\DB_CONNECT\\Vector.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# SDE feature class
sde_Bailey = sdeBase + "\\sde.SDE.Soils\\sde.SDE.land_capability_Bailey_Soils"

# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# create out table for the stats sum
outTable =  "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Bailey_Table"

# Create Identity Output Layer
id_ParcelLyr_BaileyLyr = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Bailey"

# Create Impervious Layer
Bailey_lyr = wk_memory + "Bailey_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(Bailey_lyr):
    arcpy.Delete_management(Bailey_lyr)
    print ("Deleted Bailey Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_BaileyLyr):
    arcpy.Delete_management(id_ParcelLyr_BaileyLyr)
    print ("Deleted Parcel Bailey Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Bailey, Bailey_lyr)
print ("Created feature layer of Bailey Soils")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, Bailey_lyr, id_ParcelLyr_BaileyLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_BaileyLyr, identity_layer, where_clause = "NOT CAPABILITY in ('WB', '-1', '0')")
print ("Created identity feature layer of bailey")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Square Footage all polygons")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['CAPABILITY', 'SqFt', 'PERCENT_COVERAGE_ALLOWED']) as cur:
    for row in cur:
        if row[0] != ('','WB'):
            row[1] = row[1]*row[2]
        else:
            row[1] == 0
        cur.updateRow(row)
    print("Calculated Allowed Square Footage")
    
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN")
print ("Summed Square Footage of Bailey")

## Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
fieldJoinCalc(ParcelLayer, ['APN', 'ALLOWABLE_COVERAGE_BAILEY_SQFT'], outTable, ['APN', 'SUM_SqFt'])
print ("The 'ALLOWABLE_COVERAGE_BAILEY_SQFT' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["ALLOWABLE_COVERAGE_BAILEY_SQFT"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = 0
            cursor.updateRow(row)
print ("NULL values have been updated")

Created feature layer of Bailey Soils
Starting Identity
Finished Identity
Created identity feature layer of bailey
Calculated Square Footage all polygons
Calculated Allowed Square Footage
Summed Square Footage of Bailey
Started data transfer: 2021-05-05 12:29:23
Finished data transfer: 2021-05-05 12:29:37
The 'ALLOWABLE_COVERAGE_BAILEY_SQFT' field in the parcel data has been updated
NULL values have been updated


### Update Land Cabality Field

In [24]:
# Import system modules
import arcpy
import os

# Set local variables
workspace = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb"

arcpy.env.overwriteOutput = True

# Want to join Land Capability to Parcels
targetFeatures = os.path.join(workspace, "Parcel_Master")
joinFeatures = os.path.join(workspace, "NRCS2007")

# Output will be the target features, states, with a mean city population field (mcp)
outfc = os.path.join(workspace, "spjn_Parcel_NRCS")

# Create a new fieldmappings and add the two input feature classes.
fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(targetFeatures)
fieldmappings.addTable(joinFeatures)

# get the field mappings for 
landCapFieldIndex = fieldmappings.findFieldMapIndex("Land_Capab")
fieldmap = fieldmappings.getFieldMap(landCapFieldIndex)

# Get the output field's properties as a field object
field = fieldmap.outputField

# Rename the field and pass the updated field object back into the field map
field.name = "LandCapability"
field.aliasName = "Land Capability"
field.length = 500
fieldmap.outputField = field

# Set the merge rule to join with a comma delimiter and then replace the old fieldmap in the mappings object with the updated one
fieldmap.mergeRule = "join"
fieldmap.joinDelimiter = ','
fieldmappings.replaceFieldMap(landCapFieldIndex, fieldmap)

#Run the Spatial Join tool, using the defaults for the join operation and join type
arcpy.SpatialJoin_analysis(targetFeatures, joinFeatures, outfc, "#", "#", fieldmappings)
print ("Spatial join complete.")

spjnFC = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\spjn_Parcel_NRCS"
# remove duplicate land capabilities
with arcpy.da.UpdateCursor(spjnFC, ['CAPABILITY']) as cur:
    for row in cur:
        if row[0] is not None:
            v = row[0]
            v = v.split(',')
            v = list(set(v))
            v = ','.join(v)
            row[0] = v
            cur.updateRow(row)
    print ("Duplicate values removed.")

RuntimeError: FieldMappings: Error in adding table to field mappings

### Update Impervious Coverage Field

In [3]:
import arcpy
arcpy.env.overwriteOutput = True

# SDE file path 
#sdeBase = "F:\\GIS\\DB_CONNECT\\Vector.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# SDE feature class
# sde_Impervious = sdeBase + "\\sde.SDE.Impervious\\sde.SDE.Impervious_2019"
sde_Impervious = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Impervious_2019"

# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# create out table for the stats sum
outTable =  "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Imp_Table"

# Create Identity Output Layer
id_ParcelLyr_ImperviousLyr = wk_memory + "id_Parcel_Impervious"

# Create Impervious Layer
Impervious_lyr = wk_memory + "Impervious_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(Impervious_lyr):
    arcpy.Delete_management(Impervious_lyr)
    print ("Deleted Impervious Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_ImperviousLyr):
    arcpy.Delete_management(id_ParcelLyr_ImperviousLyr)
    print ("Deleted Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Impervious, Impervious_lyr, where_clause = "Feature IN ('Building', 'Road', 'Other', 'Driveway')")
print ("Created impervious feature layer of buildings and other impervious surface")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, Impervious_lyr, id_ParcelLyr_ImperviousLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_ImperviousLyr, identity_layer, where_clause = "Feature IN ('Building', 'Road', 'Other', 'Driveway')")
print ("Created identity feature layer of buildings and other impervious surface")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Impervious Square Footage")
                                                           
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN")
print ("Summed Square Footage of Impervious")

# Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
fieldJoinCalc(ParcelLayer, ['APN', 'IMPERVIOUS_SURFACE_SQFT'], outTable, ['APN', 'SUM_SqFt'])
print ("The 'ImperviousCoverage_SqFt' field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["IMPERVIOUS_SURFACE_SQFT"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = 0
            cursor.updateRow(row)
print ("NULL values have been updated")

Deleted Out Table
Created impervious feature layer of buildings and other impervious surface
Starting Identity
Finished Identity
Created identity feature layer of buildings and other impervious surface
Calculated Impervious Square Footage
Summed Square Footage of Impervious
Started data transfer: 2021-05-19 09:44:51
Finished data transfer: 2021-05-19 09:45:05
The 'ImperviousCoverage_SqFt' field in the parcel data has been updated
NULL values have been updated


### Update TAZ Field

In [7]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TAZ):
    arcpy.Delete_management(ParcelPoint_TAZ)

print("Starting the TAZ Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TAZ, ParcelPoint_TAZ, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the TAZ Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'TAZ'], ParcelPoint_TAZ, ['APN', 'TAZ_1'])
print ("The TAZ field in the parcel data has been updated")

# change NULL values to blank values
with arcpy.da.UpdateCursor(ParcelLayer, ["TAZ"]) as cursor:
    for row in cursor:
        if row[0] == None:
            row[0] = 0
            cursor.updateRow(row)
print ("NULL values have been updated")

Starting the TAZ Spatial Join
Finished the TAZ Spatial Join
Started data transfer: 2021-05-05 12:42:28
Finished data transfer: 2021-05-05 12:42:38
The TAZ field in the parcel data has been updated
NULL values have been updated


### Update Land Capability Coverage Allowed field (based on LCV layer)

#### Pseudo Code
* Push value to Parcel_Master and Parcel_Simplified

In [ ]:
import arcpy
arcpy.env.overwriteOutput = True
def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    
    #updateFC = r"C:\Path\UpdateFeatureClass"  
    #updateFieldsList = ["JoinField", "ValueField"]
    
    # sourceFC = r"C:\Path\SourceFeatureClass"  
    #sourceFieldsList = ["JoinField", "ValueField"]  
    
    # Use list comprehension to build a dictionary from a da SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

# SDE file path 
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"
sdeCollect ="F:\\GIS\\GIS_DATA\\Collection.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# # LCV feature class
# sde_LCV = sdeCollect + "\\sde_collection.SDE.LandCapabilityWebApp\\sde_collection.SDE.Land_Capability_Verification"

# parcel feature classes
# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
# ParcelLayer = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
# ParcelSimple = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcels_Simplified"

# # create out table for the stats sum
# outTable =  "F:\\GIS\\PROJECTS\\CurrentPlanning\\LCV_WebMap\\Data\\ProjectData.gdb\\id_Parcel_LCV_Table"

# # Create Identity Output Layer
# id_ParcelLyr_LCVLyr = "F:\\GIS\\PROJECTS\\CurrentPlanning\\LCV_WebMap\\Data\\ProjectData.gdb\\id_Parcel_LCV"

# Create Impervious Layer
LCV_lyr = wk_memory + "LCV_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(LCV_lyr):
    arcpy.Delete_management(LCV_lyr)
    print ("Deleted in memory LCV Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_LCVLyr):
    arcpy.Delete_management(id_ParcelLyr_LCVLyr)
    print ("Deleted Parcel LCV Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_LCV, LCV_lyr)
print ("Created feature layer of Bailey LCV Layer")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, LCV_lyr, id_ParcelLyr_LCVLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_LCVLyr, identity_layer, where_clause = "NOT LCV_IPES in ('WB', '-1', '0','IPES')")
print ("Created identity feature layer of parcels and lcv identity output")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SQFT", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Square Footage of all polygons")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['LCV_IPES', 'SQFT', 'Allowed_Coverage']) as cur:
    for row in cur:
        if row[0] != ('','WB', 'IPES'):
            if row[0] in ('1A','1B','1C','2'):
                row[2] = row[1] * 0.01
            elif row[0] == '3':
                row[2] = row[1] * 0.05
            elif row[0] == '4':
                row[2] = row[1] * 0.2
            elif row[0] == '5':
                row[2] = row[1] * 0.25
            elif row[0] in ('6','7'):
                row[2] = row[1] * 0.3
        else:
            row[2] == 0
        cur.updateRow(row)
    print("Calculated Coverage Allowed (sq.ft.)")
    
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["Allowed_Coverage", "SUM"]], "APN")
print ("Summed Square Footage of Coverage Allowed")

edit = arcpy.da.Editor(sdeBase)
print ("edit created")
try:
    edit.startEditing()
    print("edit started")
    edit.startOperation()
    print("operation started")
    # Perform edits
    ## Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
    fieldJoinCalc(ParcelSimple, ['APN', 'COVERAGE_ALLOWED'], outTable, ['APN', 'SUM_Allowed_Coverage'])
    print("The 'COVERAGE_ALLOWED' field in the parcel simplified data has been updated")
    edit.stopOperation()
    print("operation stopped")
    edit.stopEditing(True)  ## Stop the edit session with True to save the changes
    print("edit stopped")
except Exception as err:
    print(err)
    if edit.isEditing:
        edit.stopOperation()
        print("operation stopped in except")
        edit.stopEditing(False)  ## Stop the edit session with False to abandon the changes
        print("edit stopped in except")
finally:
    # Cleanup
    arcpy.ClearWorkspaceCache_management()

### Update Land Use fields

In [4]:
import arcpy

fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
fields = ("COUNTY_LANDUSE_CODE", "COUNTY_LANDUSE_DESCRIPTION", "TRPA_LANDUSE_DESCRIPTION", 'COUNTY')

with arcpy.da.UpdateCursor(fc, fields) as cursor:
    for row in cursor:
        ctyluc = row[0]
        cty = row[3]
# set Washoe county land use code
        # set TRPA Land Use Description
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('400', '410', '440', '500', '510', '520', '630', '640', '670', '720'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '250'):
                row[2] = "Condominium"
            elif ctyluc in ('240'):
                row[2] = "Condominium Common Area"
            elif ctyluc in ('220', '230', '300', '310', '320', '330', '340', '350', '360'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('600', '620'):
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', 'PBRD'):
                row[2] = "Public Service"
            elif ctyluc in ('190'):
                row[2] = "Recreation"
            elif ctyluc in ('200'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '120', '130', '140', '150', '160', '170', '180'):
                row[2] = "Vacant"            
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        # set county land use description field
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('710'):
                row[1] = "Intracounty public utility"
            elif ctyluc == '700':
                row[1] = 'Centrally assessed public utility'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial: retail or office with Indus'
            elif ctyluc == '500':
                row[1] = 'General industrial: light indust, trucking, warehs'
            elif ctyluc == '440':
                row[1] = 'Resort commercial: ski, golf, sports, etc.'
            elif ctyluc == '430':
                row[1] = 'Commercial hotel or motel'
            elif ctyluc == '420':
                row[1] = 'Casino or hotel casino'
            elif ctyluc == '410':
                row[1] = 'Offices, professional and business, banks, etc.'
            elif ctyluc == '400':
                row[1] = 'General Commercial: retail, mixed, parking, school'
            elif ctyluc == '340':
                row[1] = 'Ten or more units'
            elif ctyluc == '330':
                row[1] = 'Five to Nine Units'
            elif ctyluc == '320':
                row[1] = 'Three or four Units'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '250':
                row[1] = 'Condo or Townhouse valued as apartment use'
            elif ctyluc == '240':
                row[1] = 'Common Area'
            elif ctyluc == '210':
                row[1] = 'Condominium or Townhouse'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Public Parks: vacant or improved'
            elif ctyluc == '170':
                row[1] = 'Other, unbuildable: roads, restrictions, terrain'
            elif ctyluc == '160':
                row[1] = 'Splinter, unbuildable: small size or shape'
            elif ctyluc == '140':
                row[1] = 'Vacant, commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant, multi-residential'
            elif ctyluc == '120':
                row[1] = 'Vacant, single family'
            elif ctyluc == '110':
                row[1] = 'Vacant, under development'
            elif ctyluc == '100':
                row[1] = 'Vacant, other or unknown'            
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# Set Carson City County Land Use Descriptions
        # set TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc in ('400', '401', '402', '403', '404', '408', '410', '411', 
                          '412', '440', '441', '460', '470', '480', '482', '490', 
                          '500', '501', '510', '511', '512', '513', '520', '521', 
                          '560', '570', '580', '582', '590', '624', '625', '694', 
                          '800', '820', '830', '840', '880', '882', '890', '920', 
                          '921', '930', '960', '980', '990'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                trpalucdesc = "Condominium"
            elif ctyluc == '970':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('240', '241', '300', '301', '310', '311', '313', '320', 
                            '321', '330', '331', '333', '340', '341', '350', '360', 
                            '370', '380', '382', '390', '698'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('190', '600', '610', '612', '613', '614', '615', '616', 
                            '618', '620', '695', '696', '697', '810'):
                row[2] = "Open Space"
            elif ctyluc in ('190', '700', '710', '711', '720', '731', '732', '733', '780', 
                            '790', '910', '922'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '201', '220', '222', '230', '231', '232', '260', 
                            '270', '280', '282', '290', '622', '692', '693'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '421', '430', '431', '432', '514'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '108', '110', '117', '120', '130', '140', '150', '160'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        # set county LUC Description
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential' 
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# update Douglas Land Use descriptions        
        #Update TRPA Land Use description
        if (row[0] != None or row[0] != "") and (row[3] == 'DG'):
            if ctyluc in ('400', '402', '410', '411', '412', 
                          '440', '460', '470', '480', '500', 
                          '510', '560', '580', '582'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '270':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('300', '310', '320', '330', '350', '390'):
                row[2] = "Multi-Family Residential"
            elif ctyluc == '190':
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', '711', '910', '980', '970'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900', '970'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '220', '230', '236', '240', '280', '282'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '117', '120', '130', '140'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        if (row[0] != None or row[0] != "" or ctyluc.isspace() != True) and (row[3] == 'DG'):
            # set County land use description
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '970':
                row[1] = 'Special Purpose Common Area'
            elif ctyluc == '910':
                row[1] = 'Cemeteries'
            elif ctyluc == '900':
                row[1] = 'Parks for Public Use'
            elif ctyluc == '711':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature Under Construction'
            elif ctyluc == '710':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature'
            elif ctyluc == '700':
                row[1] = 'Operating Communication, Transportation, and Utility Property of an Interstate or Intercounty Nature'
            elif ctyluc == '582':
                row[1] = 'Industrial with Minor Improvements - with structures insufficient to determine intended use'
            elif ctyluc == '580':
                row[1] = 'Industrial with Minor Improvements'
            elif ctyluc == '560':
                row[1] = 'Industrial Auxiliary Area'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial - retail or office use combined with Industrial use'
            elif ctyluc == '500':
                row[1] = 'General Industrial - light industry, trucking and warehousing, service, repair, etc.'
            elif ctyluc == '480':
                row[1] = 'Commercial with Minor Improvements'
            elif ctyluc == '470':
                row[1] = 'Commercial Common Area'
            elif ctyluc == '460':
                row[1] = 'Commercial Auxiliary Area'
            elif ctyluc == '450':
                row[1] = 'Golf Course'
            elif ctyluc == '440':
                row[1] = 'Commercial Recreation'
            elif ctyluc == '430':
                row[1] = 'Commercial Living Accommodations'
            elif ctyluc == '420':
                row[1] = 'Casino or Hotel Casino'
            elif ctyluc == '410':
                row[1] = 'Offices, Professional and Business Services'
            elif ctyluc == '402':
                row[1] = 'Parking and/or Parking Structures'
            elif ctyluc == '400':
                row[1] = 'General Commercial'
            elif ctyluc == '390':
                row[1] = 'Mixed Use with Multi-Family Residential as primary use'
            elif ctyluc == '382':
                row[1] = 'Multi-Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '380':
                row[1] = 'Multi-Family Residential with Minor Improvements'
            elif ctyluc == '370':
                row[1] = 'Multi-Family Residential Common Area'
            elif ctyluc == '360':
                row[1] = 'Multi-Family Residential Auxiliary Area'
            elif ctyluc == '350':
                row[1] = 'Manufactured Home Park - Ten or More Manufactured Home Units'
            elif ctyluc == '341':
                row[1] = 'Five or More Units - High Rise Under Construction'
            elif ctyluc == '340':
                row[1] = 'Five or More Units - High Rise'
            elif ctyluc == '333':
                row[1] = 'Exempt or Partially Exempt Apartment Building'
            elif ctyluc == '331':
                row[1] = 'Five or More Units - Low Rise Under Construction'
            elif ctyluc == '330':
                row[1] = 'Five or More Units - Low Rise'
            elif ctyluc == '321':
                row[1] = 'Three to Four Units Under Construction'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '313':
                row[1] = 'Multi-Family Residence with Manufactured Home Conversion'
            elif ctyluc == '311':
                row[1] = 'Two Single Family Units Under Construction'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '301':
                row[1] = 'Duplex Under Construction'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '290':
                row[1] = 'Mixed Use with Single Family Residential as primary use'
            elif ctyluc == '282':
                row[1] = 'Single Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '270':
                row[1] = 'Single Family Residential Common Area'
            elif ctyluc == '260':
                row[1] = 'Single Family Residential Auxiliary Area'
            elif ctyluc == '240':
                row[1] = 'Individual Residential Unit - Townhouse or Row House'
            elif ctyluc == '236':
                row[1] = 'Personal Property Manufactured Home Secured'
            elif ctyluc == '233':
                row[1] = 'Secured Manufactured Home with Site Built Additions (Not Converted)'
            elif ctyluc == '232':
                row[1] = 'Manufactured Home - Unsecured with Site Built Additions'
            elif ctyluc == '231':
                row[1] = 'Manufacture Home Conversions Pending'
            elif ctyluc == '230':
                row[1] = 'Personal Property Manufactured Home on the Unsecured Roll'
            elif ctyluc == '222':
                row[1] = 'Manufactured Home (Converted) with Site Built Additions'
            elif ctyluc == '220':
                row[1] = 'Manufactured Home Converted to Real Property'
            elif ctyluc == '211':
                row[1] = 'Individual Unit in a Multiple Unit Building Under Construction'
            elif ctyluc == '210':
                row[1] = 'Individual Unit in a Multiple Unit Building'
            elif ctyluc == '201':
                row[1] = 'Single Family Residence Under Construction'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '150':
                row[1] = 'Vacant - Industrial'
            elif ctyluc == '140':
                row[1] = 'Vacant - Commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant - Multi-Residential'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential'
            elif ctyluc == '117':
                row[1] = 'Vacant - Roads/Easements'
            elif ctyluc == '110':
                row[1] = 'Vacant - Splinter and Other Unbuildable'
            elif ctyluc == '108':
                row[1] = 'Vacant - Patented Mining Claim, Not Mined'
            elif ctyluc == '100':
                row[1] = 'Vacant - Unknown/Other'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# Set El Dorado County Land Use Description fields
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc in ('03', '29', '31', '32', '34', '36', '37', '38', '39', '41', '42', '43', '44', '45', '46', '47', '48', 
                          '65', '67', '68', '82', '91', '93'):
                row[2] = "Commercial"
            elif ctyluc == '14':
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('01', '07', '12', '13', '16', '18', '19', '28', '35'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('25', '26', '50', '51', '52', '55', '56', '60', '70', '75', '79'):
                row[2] = "Open Space"
            elif ctyluc in ('90', '92', '94', '96', '97', '98', '99'):
                row[2] = "Public Service"
            elif ctyluc in ('61', '62', '63', '64'):
                row[2] = "Recreation"
            elif ctyluc in ('06', '11', '15', '22', '23'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('33', '80', '81'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '02', '05', '17', '21', '24', '30', '40'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
# set county land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc == '98':
                row[1] = 'DEV MSC FIRE SUPPRESSION FACILITIES'
            elif ctyluc == '96':
                row[1] = 'DEV MSC CEMETERIES'
            elif ctyluc == '94':
                row[1] = 'DEV MSC SCHOOLS - LARGE (101+ STUDENTS)'
            elif ctyluc == '93':
                row[1] = 'DEV MSC SCHOOLS - MEDIUM (13-100 STUDENTS)'
            elif ctyluc == '92':
                row[1] = 'DEV MSC SCHOOLS - SMALL (1-12 STUDENTS)'
            elif ctyluc == '90':
                row[1] = 'UTL IND PUBLIC UTILITY (ON STATE ASSESSED ROLL)'
            elif ctyluc == '84':
                row[1] = 'DEV MSC TEMPORARY USE CODE FOR PROJECT 184'
            elif ctyluc == '82':
                row[1] = 'DEV COM PARKING LOT'
            elif ctyluc == '81':
                row[1] = 'DEV MSC UNDERLYING INTEREST IN TIME SHARE PROJ'
            elif ctyluc == '79':
                row[1] = 'RLU MSC ENV. SENSITIVE LAND - RESTRICTED USE'
            elif ctyluc == '68':
                row[1] = 'DEV COM MARINAS'
            elif ctyluc == '65':
                row[1] = 'DEV COM RESTAURANT'
            elif ctyluc == '64':
                row[1] = 'DEV MSC SKI RESORTS'
            elif ctyluc == '63':
                row[1] = 'DEV MSC CAMPGROUNDS'
            elif ctyluc == '62':
                row[1] = 'DEV MSC COMMUNITY ORIENTED FACILITIES'
            elif ctyluc == '61':
                row[1] = 'DEV MSC MISC. IMPROVED RECREATIONAL'
            elif ctyluc == '60':
                row[1] = 'VAC MSC VACANT RECREATIONAL LAND'
            elif ctyluc == '50':
                row[1] = 'TPZ MSC TIMBER PRESERVE ZONING - ACTIVE'
            elif ctyluc == '48':
                row[1] = 'DEV IND OFFICES'
            elif ctyluc == '47':
                row[1] = 'DEV IND HOSPITALS & CONVALESCENT HOSPITALS'
            elif ctyluc == '46':
                row[1] = 'DEV IND MEDICAL/DENTAL/VET OFFICES'
            elif ctyluc == '45':
                row[1] = 'DEV IND LIGHT MANUFACTURING'
            elif ctyluc == '43':
                row[1] = 'DEV IND WAREHOUSES'
            elif ctyluc == '42':
                row[1] = 'DEV IND MINI-WAREHOUSES (MINI-STORAGE)'
            elif ctyluc == '41':
                row[1] = 'DEV IND MISC. IMPROVED INDUSTRIAL PROPERTY'
            elif ctyluc == '40':
                row[1] = 'VAC IND VACANT INDUSTRIAL LAND'
            elif ctyluc == '39':
                row[1] = 'DEV COM SUPERMARKETS'
            elif ctyluc == '38':
                row[1] = 'DEV COM RETAIL STORES >15,000 SQ. FT.'
            elif ctyluc == '37':
                row[1] = 'DEV COM RETAIL STORES 5,001-15,000 SQ. FT.'
            elif ctyluc == '36':
                row[1] = 'DEV COM RETAIL STORES <=5,000 SQ. FT.'
            elif ctyluc == '35':
                row[1] = 'DEV COM MOBILE HOME PARKS'
            elif ctyluc == '34':
                row[1] = 'DEV COM SERVICE STATION'
            elif ctyluc == '33':
                row[1] = 'DEV COM MOTEL, HOTEL'
            elif ctyluc == '31':
                row[1] = 'DEV COM MISC. IMPROVED COMMERCIAL'
            elif ctyluc == '30':
                row[1] = 'VAC COM VACANT COMMERCIAL LAND'
            elif ctyluc == '29':
                row[1] = 'DEV MSC RURAL NON-RES. IMPROVEMENT 2.51-20.0 AC.'
            elif ctyluc == '26':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - NON-RENEWAL'
            elif ctyluc == '25':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - CLCA (ACTIVE)'
            elif ctyluc == '24':
                row[1] = 'VAC RES RURAL RES. LAND 20+ MINOR NON-RES IMPR'
            elif ctyluc == '23':
                row[1] = 'DEV RES RURAL RES. 20+ AC. 1 RES. UNIT'
            elif ctyluc == '22':
                row[1] = 'DEV RES RURAL RES. 2.51-20.0 AC. 1 SF UNIT'
            elif ctyluc == '21':
                row[1] = 'VAC RES VAC RURAL RES LAND 2.51-20.0 AC. 1 UNIT'
            elif ctyluc == '17':
                row[1] = 'VAC MSC SUBJ. TO OPEN SPACE CONTRACT (NOT CLCA)'
            elif ctyluc == '16':
                row[1] = 'DEV RES MOBILE HOME ON RENTED LAND'
            elif ctyluc == '15':
                row[1] = 'DEV RES RESIDENCE ON LEASED LAND'
            elif ctyluc == '14':
                row[1] = 'DEV MFR CONDOMINIUMS & TOWNHOUSES'
            elif ctyluc == '13':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 4+ UNITS'
            elif ctyluc == '12':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 2-3 UNITS'
            elif ctyluc == '11':
                row[1] = 'DEV RES SINGLE FAM. RES. <=2.5 AC.(INC. MAN. HMS'
            elif ctyluc == '07':
                row[1] = 'DEV MFR RETIREMENT HOUSING'
            elif ctyluc == '05':
                row[1] = 'VAC MFR VACANT MULTI-RES. LAND 4+ UNITS ALLOWED'
            elif ctyluc == '03':
                row[1] = 'DEV COM PLACE OF WORSHIP'
            elif ctyluc == '02':
                row[1] = 'VAC RES NON-RES. IMPROVEMENTS <=2.5 AC.'
            elif ctyluc == '00':
                row[1] = 'VAC RES VACANT RES. LAND <=2.5 AC. 1-3 UNITS'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)

# set Placer TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc in ('07', '11', '12', '13', '14', '15', '17', '19', '21', '22', '23', 
                          '24', '25', '26', '27', '29', '31', '32', '36', '37', '38', 
                          '39', '62', '63', '71', '88'):
                row[2] = "Commercial"
            elif ctyluc == ('04', '19'):
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('02', '03', '04', '05', '09', '28'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('55', '56', '60', '87', '90'):
                row[2] = "Open Space"
            elif ctyluc in ('72', '76', '77', '81'):
                row[2] = "Public Service"
            elif ctyluc in ('65', '66', '67', '68', '69'):
                row[2] = "Recreation"
            elif ctyluc in ('01', '08', '16'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('06', '18', '64'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '10', '20', '30'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
# set Placer county land use description 
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            # set county land use description
            if ctyluc == '90':
                row[1] = 'GREENBELT'
            elif ctyluc == '89':
                row[1] = 'COMMON AREA'
            elif ctyluc == '88':
                row[1] = 'HIGHWAYS, ROADS, STREETS'
            elif ctyluc == '87':
                row[1] = 'RIVERS, LAKES, RESERVOIR, CANAL'
            elif ctyluc == '81':
                row[1] = 'UTILITIES, PUBLIC & PRIVATE'
            elif ctyluc == '77':
                row[1] = 'CEMETERIES'
            elif ctyluc == '76':
                row[1] = 'MISC. PUBLIC BUILDINGS'
            elif ctyluc == '72':
                row[1] = 'SCHOOLS'
            elif ctyluc == '71':
                row[1] = 'CHURCHES'
            elif ctyluc == '69':
                row[1] = 'MISCELLANEOUS RECREATIONAL'
            elif ctyluc == '68':
                row[1] = 'CAMPS & PARKS, GENERAL'
            elif ctyluc == '67':
                row[1] = 'SKI FACILITY'
            elif ctyluc == '66':
                row[1] = 'GOLF COURSE'
            elif ctyluc == '65':
                row[1] = 'TENNIS, SWIMMING CLUBS'
            elif ctyluc == '64':
                row[1] = 'LODGES, HALLS'
            elif ctyluc == '63':
                row[1] = 'MARINA, PIER'
            elif ctyluc == '62':
                row[1] = 'THEATER, BOWLING ALLEY'
            elif ctyluc == '61':
                row[1] = 'NON-PROFIT CAMPS/PARKS'
            elif ctyluc == '60':
                row[1] = 'CONSERVATION EASEMENT RESTRICTIONS'
            elif ctyluc == '56':
                row[1] = 'TIMBERLAND, ZONED TPZ'
            elif ctyluc == '55':
                row[1] = 'TIMBERLAND, UNRESTRICTED'
            elif ctyluc == '39':
                row[1] = 'MISCELLANEOUS INDUSTRIAL'
            elif ctyluc == '38':
                row[1] = 'WAREHOUSE'
            elif ctyluc == '37':
                row[1] = 'MINI-STORAGE, COVERED STORAGE'
            elif ctyluc == '36':
                row[1] = 'UNCOVERED STORAGE, WRECKING YARD'
            elif ctyluc == '32':
                row[1] = 'HEAVY INDUSTRIAL'
            elif ctyluc == '31':
                row[1] = 'LIGHT INDUSTRIAL'
            elif ctyluc == '30':
                row[1] = 'VACANT INDUSTRIAL'
            elif ctyluc == '29':
                row[1] = "MISCELLANEOUS COMM'L"
            elif ctyluc == '28':
                row[1] = 'MOBILE HOME PARK'
            elif ctyluc == '27':
                row[1] = 'PARKING LOTS'
            elif ctyluc == '26':
                row[1] = 'AUTO SALES, REPAIR'
            elif ctyluc == '25':
                row[1] = 'SERVICE STATION'
            elif ctyluc == '24':
                row[1] = 'MINI-MARKET WITH GAS'
            elif ctyluc == '23':
                row[1] = "BANKS, S&L'S, CREDIT UNION"
            elif ctyluc == '22':
                row[1] = 'FAST FOOD RESTAURANT'
            elif ctyluc == '21':
                row[1] = 'RESTAURANTS, COCKTAIL LOUNGES'
            elif ctyluc == '20':
                row[1] = 'VACANT, COMMERCIAL'
            elif ctyluc == '19':
                row[1] = 'OFFICE MEDICAL/DENTAL'
            elif ctyluc == '18':
                row[1] = 'HOTELS, MOTELS, RESORTS'
            elif ctyluc == '17':
                row[1] = 'OFFICE GENERAL'
            elif ctyluc == '16':
                row[1] = 'RESIDENCE ON COMMERCIAL LAND'
            elif ctyluc == '15':
                row[1] = 'SHOPPING CENTER'
            elif ctyluc == '14':
                row[1] = 'OFFICE CONDO'
            elif ctyluc == '13':
                row[1] = 'MINI-MARKETS, NO GAS'
            elif ctyluc == '12':
                row[1] = 'SUBURBAN STORE'
            elif ctyluc == '11':
                row[1] = 'COMMERCIAL STORE'
            elif ctyluc == '10':
                row[1] = 'VACANT, SUBDIVIDED RESIDENTIAL'
            elif ctyluc == '09':
                row[1] = 'MOBILE HOME IN M H PARK'
            elif ctyluc == '08':
                row[1] = 'MOBILE HOME OUTSIDE OF PARK'
            elif ctyluc == '07':
                row[1] = 'RESIDENTIAL, AUXILIARY IMP'
            elif ctyluc == '06':
                row[1] = 'TIMESHARES'
            elif ctyluc == '05':
                row[1] = 'APARTMENTS, 4 UNITS OR MORE'
            elif ctyluc == '04':
                row[1] = 'SINGLE FAM RES, CONDO'
            elif ctyluc == '03':
                row[1] = '3 SINGLE FAM RES, TRIPLEX'
            elif ctyluc == '02':
                row[1] = '2 SINGLE FAM RES, DUPLEX'
            elif ctyluc == '01':
                row[1] = 'SINGLE FAM RES, HALF PLEX'
            elif ctyluc == '00':
                row[1] = 'VACANT, ALL TYPES-NOT ASGND'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
    print ("Updated Land Use Description fields.")
    
# lists all field names and their index numbers
#field_names = [field.name for field in arcpy.ListFields(ParcelLayer)]
# change NULL values to blank values
#with arcpy.da.UpdateCursor(ParcelLayer, field_names) as cursor:
    #for row in cursor:
        #countycode = row[24]
        #if (countycode is None or countycode == "" or countycode.isspace() == True):
            #row[24] = ""
        #countylu = row[25]
        #if (countylu is None or countylu == "" or countylu.isspace() == True):
            #row[25] = ""
        #trpalu = row[26]
        #if (trpalu is None or trpalu == "" or trpalu.isspace() == True):
            #row[26] = ""
            #cursor.updateRow(row)
#print ("NULL values have been updated")

Updated Land Use Description fields.


### Update Land Use fields with Collection SDE

#### Identify new county values
##### save list of new county values

#### Grab Collection SDE edited values
##### check to see if any edited values conflict with new county values

#### Compile of list master TRPA Landuse Values
##### Move to Parcel Master SDE
##### Move to Parcel Edit and Parcel Master Staging 


#### Pseudo Code

In [1]:
#Set variables
ParcelLayer = Workspace layer

OldParcelLayer = parcel master in sde 


# Final 3 Feature Classes
CountyChange =

# In memory layers

NewParcels_OldParcels = 
LandUseChange = 
NoLUChange =
TrueChange =
TRPAChange =

NewParcels_CollectionParcels =
NoChangeJoin =

# Spatial join between old parcel layer and new parcel layer
arcpy.SpatialJoin_analysis(ParcelLayer, OldParcelLayer, NewParcels_OldParcels, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Parcel Spatial Join")

# Find values that changed between last update and this update
field_names = ['TRPA_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION_1', 'DUPLICATE']
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)
with arcpy.da.UpdateCursor(NewParcels_OldParcels, field_names) as cursor:
    for row in cursor:
        # set fields
        landuse1 = row[0]
        landuse2 = row[1]
        if landuse1==landuse2:
            row[2] = 0
        else:
            row[1] = 1
            
# Select features that are different and export to temporary layer
arcpy.Select_analysis(NewParcels_OldParcels, LandUseChange, '"DUPLICATE" = 1')

# Switch selection to get values that didn't change
arcpy.SelectLayerByAttribute_management(NewParcels_OldParcels, "SWITCH_SELECTION")
# Export to temporary layer
arcpy.MakeFeatureLayer_management(NewParcels_OldParcels, NoLUChange)
# Clear Selection
arcpy.SelectLayerByAttribute_management(NewParcels_OldParcels, "CLEAR_SELECTION")

# Separate true land use change from TRPA change
field_names = ['COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_CODE_1', 'DUPLICATE']
with arcpy.da.UpdateCursor(NewParcels_OldParcels, field_names) as cursor:
    for row in cursor:
        # set fields
        countyuse1 = row[0]
        countyuse2 = row[1]
        if countyuse1==countyuse2:
            row[2] = 0
        else:
            row[1] = 1
            
# Select features with true change from the county and export as final feature class
arcpy.Select_analysis(LandUseChange, CountyChange, '"DUPLICATE" = 1')

# Switch selection to get values that TRPA changed not the county
arcpy.SelectLayerByAttribute_management(LandUseChange, "SWITCH_SELECTION")
# Export to temporary layer
arcpy.MakeFeatureLayer_management(LandUseChange, TRPAChange)
# Clear Selection
arcpy.SelectLayerByAttribute_management(LandUseChange, "CLEAR_SELECTION")


# Spatial join between parcel layer and collection parcel layer
arcpy.SpatialJoin_analysis(ParcelLayer, CollectionEditLayer, NewParcels_CollectionParcels, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Parcel Spatial Join")

# Compare no change parcels to parcel edit
arcpy.AddJoin_management(NoLUChange, CollectionEditLayer, NoChangeJoin, "APN")

field_names = ['TRPA_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION_2', 'DUPLICATE']
with arcpy.da.UpdateCursor(NoChangeJoin, field_names) as cursor:
    for row in cursor:
        # set fields
        landuse1 = row[0]
        landuse2 = row[1]
        if landuse1==landuse2:
            row[2] = 0
        else:
            row[1] = 1

# Export parcels that didn't change as final no change table
arcpy.Select_analysis(LandUseChange, CountyChange, '"DUPLICATE" = 1')

# Switch selection and export parcels that changed to temporary layer
arcpy.SelectLayerByAttribute_management(LandUseChange, "SWITCH_SELECTION")
# Export to temporary layer
arcpy.MakeFeatureLayer_management(LandUseChange, TRPAChange)
# Clear Selection
arcpy.SelectLayerByAttribute_management(LandUseChange, "CLEAR_SELECTION")

# Append TRPA changes and parcel edit changes. Export to final table

# Manually check CollectionChange feature class and CountyChange feature class

# Update CollectionChange list in Parcel Master Layer

SyntaxError: invalid syntax (<ipython-input-1-b24aaf86f9f6>, line 2)

In [3]:
#Set variables
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
OldParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\Parcel_Master"


# Final 3 Feature Classes
Parcels_Join = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\Parcels_Join"
CountyChange = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\CountyChange"
TRPAChange = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\TRPAChange"
LandUseChange = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\LandUseChange"
NoLUChange = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb\\NoLUChange"

# Change layers
# wk_memory = "in_memory" + "\\"
# LandUseChange = wk_memory + "LandUseChange"
# NoLUChange = wk_memory + "NoLUChange"

# Spatial join between old parcel layer and new parcel layer
arcpy.SpatialJoin_analysis(ParcelLayer, OldParcelLayer, Parcels_Join, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Parcel Spatial Join")

# Find values that changed between last update and this update
field_names = ['TRPA_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION_1', 'DUPLICATE']
with arcpy.da.UpdateCursor(Parcels_Join, field_names) as cursor:
    for row in cursor:
        # set fields
        landuse1 = row[0]
        landuse2 = row[1]
        if landuse1==landuse2:
            row[2] = 0
        else:
            row[2] = 1

print ("Field Calculation Complete")
            
            
 # Execute FeatureClassToFeatureClass
arcpy.env.workspace = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb"
arcpy.FeatureClassToFeatureClass_conversion("Parcels_Join", 
                                            "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb", 
                                            "LandUseChange",
                                            '"DUPLICATE" = 1')
print ("Land Use Change Feature Class Created")
arcpy.FeatureClassToFeatureClass_conversion("Parcels_Join", 
                                            "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb", 
                                            "NoLUChange",
                                            '"DUPLICATE" = 0')
print ("No Land Use Change Feature Class Created")           
            
# # Select features that are different and export to temporary layer
# arcpy.Select_analysis(Parcels_Join, LandUseChange, '"DUPLICATE" = 1')

# # Switch selection to get values that didn't change
# arcpy.SelectLayerByAttribute_management(Parcels_Join, "SWITCH_SELECTION")
# # Export to temporary layer
# arcpy.MakeFeatureLayer_management(Parcels_Join, NoLUChange)
# # Clear Selection
# arcpy.SelectLayerByAttribute_management(Parcels_Join, "CLEAR_SELECTION")


arcpy.AddField_management(LandUseChange, "DUPLICATE2", "TEXT", 100)

# Separate true land use change from TRPA change
field_names = ['COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_CODE_1', 'DUPLICATE2']
with arcpy.da.UpdateCursor(LandUseChange, field_names) as cursor:
    for row in cursor:
        # set fields
        countyuse1 = row[0]
        countyuse2 = row[1]
        if countyuse1==countyuse2:
            row[2] = 0
        else:
            row[2] = 1
            
print ("Field Calculation Complete")

 # Execute FeatureClassToFeatureClass
arcpy.env.workspace = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb"
arcpy.FeatureClassToFeatureClass_conversion("LandUseChange", 
                                            "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb", 
                                            "CountyChange",
                                            '"DUPLICATE2" = 1')
print ("County Change Feature Class Created")
arcpy.FeatureClassToFeatureClass_conversion("LandUseChange", 
                                            "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Land_Use.gdb", 
                                            "TRPAChange",
                                            '"DUPLICATE2" = 0 AND "DUPLICATE" = 1')
print ("TRPA Change Feature Class Created")    
            
# # Select features with true change from the county and export as final feature class
# arcpy.Select_analysis(LandUseChange, CountyChange, '"DUPLICATE" = 1')

# # Switch selection to get values that TRPA changed not the county
# arcpy.SelectLayerByAttribute_management(LandUseChange, "SWITCH_SELECTION")
# # Export to temporary layer
# arcpy.MakeFeatureLayer_management(LandUseChange, TRPAChange)
# # Clear Selection
# arcpy.SelectLayerByAttribute_management(LandUseChange, "CLEAR_SELECTION")

Finished the Parcel Spatial Join
Field Calculation Complete
Land Use Change Feature Class Created
No Land Use Change Feature Class Created
Field Calculation Complete


ExecuteError: ERROR 160195: An invalid SQL statement was used.
Failed to execute (FeatureClassToFeatureClass).


### Update Ownership Type field with Collection SDE

## Create Workspace Tables

In [ ]:
import arcpy
table ="F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
field_names = [field.name for field in arcpy.ListFields(simplifiedFeatures)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

In [ ]:
import arcpy

def getFieldMappings(fc_in, mapping_list):
    field_mappings = arcpy.FieldMappings()

    for in_field, out_field, out_type in mapping_list:
        field_map = arcpy.FieldMap()
        field_map.addInputField(fc_in, in_field)
        field = field_map.outputField
        field.name = out_field
        field.type = out_type
        field_map.outputField = field
        field_mappings.addFieldMap(field_map)
        print("added {} field map object".format(field.name))
        del field, field_map

    return field_mappings

# input table and output path
table = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
outpath = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb"

# list of all Parcel fields 
fieldListMaster = ('APN', 'PPNO', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME', 'STR_SUFFIX', 
               'APO_ADDRESS', 'PSTL_TOWN', 'PSTL_STATE', 'PSTL_ZIP5', 'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 
               'MAIL_ADD1', 'MAIL_ADD2', 'MAIL_CITY', 'MAIL_STATE', 'MAIL_ZIP5', 'JURISDICTION', 'COUNTY', 
               'OWNERSHIP_TYPE', 'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
               'REGIONAL_LANDUSE', 'UNITS', 'BEDROOMS', 'BATHROOMS', 'ALLOWABLE_COVERAGE_BAILEY_SQFT', 
               'IMPERVIOUS_SURFACE_SQFT', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 'WATERSHED_NUMBER', 'WATERSHED_NAME', 
               'PRIORITY_WATERSHED', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 'AS_LANDVALUE', 'AS_IMPROVALUE', 
               'AS_SUM', 'TAX_LANDVALUE', 'TAX_IMPROVALUE', 'TAX_SUM', 'TAX_YEAR', 'PLAN_ID', 'PLAN_NAME', 
               'ZONING_ID', 'ZONING_DESCRIPTION', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'INDEX_1987', 
               'LOCAL_PLAN_HYPERLINK', 'DESIGN_GUIDELINES_HYPERLINK', 'INDEX_1987_HYPERLINK', 
               'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@')

#Export Parcel_Master table
table_out = "Parcel_Master_Table"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

master_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "HSE_NUMBR", "Short integer"),
               (fieldListMaster[3], "UNIT_NUMBR", "Text"),
               (fieldListMaster[4], "STR_DIR", "Text"),
               (fieldListMaster[5], "STR_NAME", "Text"),
               (fieldListMaster[6], "STR_SUFFIX", "Text"),
               (fieldListMaster[7], "APO_ADDRESS", "Text"),
               (fieldListMaster[8], "PSTL_TOWN", "Text"),
               (fieldListMaster[9], "PSTL_STATE", "Text"),
               (fieldListMaster[10], "PSTL_ZIP5", "Text"),
               (fieldListMaster[11], "OWN_FIRST", "Text"),
               (fieldListMaster[12], "OWN_LAST", "Text"),
               (fieldListMaster[13], "OWN_FULL", "Text"),
               (fieldListMaster[14], "MAIL_ADD1", "Text"),
               (fieldListMaster[15], "MAIL_ADD2", "Text"),
               (fieldListMaster[16], "MAIL_CITY", "Text"),
               (fieldListMaster[17], "MAIL_STATE", "Text"),
               (fieldListMaster[18], "MAIL_ZIP5", "Text"),
               (fieldListMaster[19], "JURISDICTION", "Text"),
               (fieldListMaster[20], "COUNTY", "Text"),
               (fieldListMaster[21], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[22], "COUNTY_LANDUSE_CODE", "Text"),
               (fieldListMaster[23], "COUNTY_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[24], "TRPA_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[25], "REGIONAL_LANDUSE", "Text"),
               (fieldListMaster[26], "UNITS","Text"),
               (fieldListMaster[27], "BEDROOMS", "Text"),
               (fieldListMaster[28], "BATHROOMS", "Text"),
               (fieldListMaster[29], "ALLOWABLE_COVERAGE_BAILEY_SQFT", "Double"),
               (fieldListMaster[30], "IMPERVIOUS_SURFACE_SQFT", "Double"),
               (fieldListMaster[31], "SOIL_1974", "Text"),
               (fieldListMaster[32], "SOIL_2003", "Text"),
               (fieldListMaster[33], "HRA_NAME", "Text"),
               (fieldListMaster[34], "WATERSHED_NUMBER", "Short integer"),
               (fieldListMaster[35], "WATERSHED_NAME", "Text"),
               (fieldListMaster[36], "PRIORITY_WATERSHED", "Text"),
               (fieldListMaster[37], "FIREPD", "Text"),
               (fieldListMaster[38], "WITHIN_TRPA_BNDY", "Short integer"),
               (fieldListMaster[39], "LITTORAL", "Short integer"),
               (fieldListMaster[40], "AS_LANDVALUE", "Long integer"),
               (fieldListMaster[41], "AS_IMPROVALUE", "Long integer"),
               (fieldListMaster[42], "AS_SUM", "Long integer"),
               (fieldListMaster[43], "TAX_LANDVALUE", "Long integer"),
               (fieldListMaster[44], "TAX_IMPROVALUE", "Long integer"),
               (fieldListMaster[45], "TAX_SUM", "Long integer"),
               (fieldListMaster[46], "TAX_YEAR", "Text"),
               (fieldListMaster[47], "PLAN_ID", "Text"),
               (fieldListMaster[48], "PLAN_NAME", "Text"),
               (fieldListMaster[49], "ZONING_ID", "Text"),
               (fieldListMaster[50], "ZONING_DESCRIPTION", "Text"),
               (fieldListMaster[51], "TOWN_CENTER", "Text"),
               (fieldListMaster[52], "LOCATION_TO_TOWNCENTER", "Text"),
               (fieldListMaster[53], "INDEX_1987", "Text"),
               (fieldListMaster[54], "LOCAL_PLAN_HYPERLINK", "Text"),
               (fieldListMaster[55], "DESIGN_GUIDELINES_HYPERLINK", "Text"),
               (fieldListMaster[56], "INDEX_1987_HYPERLINK", "Text"),
               (fieldListMaster[57], "PARCEL_ACRES", "Double"),
               (fieldListMaster[58], "PARCEL_SQFT", "Double")]
                              

mapped = getFieldMappings(table, master_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


#Export Parcel_Address Table
table_out = "Parcel_Address"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

address_dict = [(fieldListMaster[0], "APN", "Text"),
                (fieldListMaster[1], "PPNO", "Double"),
                (fieldListMaster[19], "JURISDICTION", "Text"),
                (fieldListMaster[2], "HSE_NUMBR", "Short integer"),
                (fieldListMaster[3], "UNIT_NUMBR", "Text"),
                (fieldListMaster[4], "STR_DIR", "Text"),
                (fieldListMaster[5], "STR_NAME", "Text"),
                (fieldListMaster[6], "STR_SUFFIX", "Text"),
                (fieldListMaster[7], "APO_ADDRESS", "Text"),
                (fieldListMaster[8], "PSTL_TOWN", "Text"),
                (fieldListMaster[9], "PSTL_STATE", "Text"),
                (fieldListMaster[10], "PSTL_ZIP5", "Text")]

mapped = getFieldMappings(table, address_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


#Export Parcel_Owner Table
table_out = "Parcel_Owner"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

owner_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[19], "JURISDICTION", "Text"),
               (fieldListMaster[11], "OWN_FIRST", "Text"),
               (fieldListMaster[12], "OWN_LAST", "Text"),
               (fieldListMaster[13], "OWN_FULL", "Text"),
               (fieldListMaster[14], "MAIL_ADD1", "Text"),
               (fieldListMaster[15], "MAIL_ADD2", "Text"),
               (fieldListMaster[16], "MAIL_CITY", "Text"),
               (fieldListMaster[17], "MAIL_STATE", "Text"),
               (fieldListMaster[18], "MAIL_ZIP5", "Text")]

mapped = getFieldMappings(table, owner_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


# Export Parcel_Value Table
table_out = "Parcel_Value"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

value_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[19], "JURISDICTION", "Text"),
               (fieldListMaster[40], "AS_LANDVALUE", "Long integer"),
               (fieldListMaster[41], "AS_IMPROVALUE", "Long integer"),
               (fieldListMaster[42], "AS_SUM", "Long integer"),
               (fieldListMaster[43], "TAX_LANDVALUE", "Long integer"),
               (fieldListMaster[44], "TAX_IMPROVALUE", "Long integer"),
               (fieldListMaster[45], "TAX_SUM", "Long integer"),
               (fieldListMaster[46], "TAX_YEAR", "Text"),
               (fieldListMaster[7], "APO_ADDRESS", "Text"),
               (fieldListMaster[8], "PSTL_TOWN", "Text"),
               (fieldListMaster[9], "PSTL_STATE", "Text"),
               (fieldListMaster[10], "PSTL_ZIP5", "Text"),
               (fieldListMaster[16], "MAIL_CITY", "Text"),
               (fieldListMaster[17], "MAIL_STATE", "Text"),
               (fieldListMaster[18], "MAIL_ZIP5", "Text"),
               (fieldListMaster[21], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[57], "PARCEL_ACRES", "Double"),
               (fieldListMaster[58], "PARCEL_SQFT", "Double")]

mapped = getFieldMappings(table, value_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


# Export Parcel_APO Table
table_out = "Parcel_APO"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

apo_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[19], "JURISDICTION", "Text"),
               (fieldListMaster[13], "OWN_FULL", "Text"),
               (fieldListMaster[7], "APO_ADDRESS", "Text"),
               (fieldListMaster[21], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[22], "COUNTY_LANDUSE_CODE", "Text"),
               (fieldListMaster[23], "COUNTY_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[24], "TRPA_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[31], "SOIL_1974", "Text"),
               (fieldListMaster[32], "SOIL_2003", "Text"),
               (fieldListMaster[33], "HRA_NAME", "Text"),
               (fieldListMaster[34], "WATERSHED_NUMBER", "Short integer"),
               (fieldListMaster[35], "WATERSHED_NAME", "Text"),
               (fieldListMaster[36], "PRIORITY_WATERSHED", "Text"),
               (fieldListMaster[47], "PLAN_ID", "Text"),
               (fieldListMaster[48], "PLAN_NAME", "Text"),
               (fieldListMaster[37], "FIREPD", "Text"),
               (fieldListMaster[38], "WITHIN_TRPA_BNDY", "Short integer"),
               (fieldListMaster[39], "LITTORAL", "Short integer"),
               (fieldListMaster[43], "PARCEL_ACRES", "Double"),
               (fieldListMaster[44], "PARCEL_SQFT", "Double"),]

mapped = getFieldMappings(table, apo_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))

## QA/QC

### Create list of new and obsolete APNs

In [ ]:
import arcpy, sys, datetime, os, traceback, csv, numpy, pandas as pd

#Set variables
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
OldParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder_old + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
Join = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Delete_Join"
Join2 = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Delete_Join2"
FinalObs = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Delete_Final"
FinalNew = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Delete_Final2"
joinfield = "APN"
ObsoleteCSV = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\ObsoleteAPNs.csv"
NewCSV = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\NewAPNs.csv"
arcpy.env.workspace = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb"

# In memory layers
wk_memory = "in_memory" + "\\"
ParcelJoin = wk_memory + "\\ParcelJoin"
ParcelJoin2 = wk_memory + "\\ParcelJoin2"

# # delete feature class if it exists already
# if arcpy.Exists(ParcelJoin):
#     arcpy.Delete_management(ParcelJoin)
    
# # delete feature class if it exists already
# if arcpy.Exists(ParcelJoin2):
#     arcpy.Delete_management(ParcelJoin2)
    
print ("Finished setting variables")



##Obsolete Parcels##
    
# Create a feature layer from featureclass
arcpy.MakeFeatureLayer_management (ParcelLayer, ParcelJoin)
print ("Made feature layer")

# Join between old parcel layer and new parcel layer
parcel_join = arcpy.AddJoin_management(ParcelJoin, joinfield, OldParcelLayer, joinfield)
print ("Finished Join")

# Write the joined features to a new featureclass
arcpy.CopyFeatures_management(ParcelJoin, Join)

# Find values that changed between last update and this update
field_names = ['Parcel_Master_APN_1', 'Parcel_Master_DUPLICATE']
# lists the index of the field names in parcel out data
#for index, field in enumerate(field_names):
    #print (index, field)
with arcpy.da.UpdateCursor(Join, field_names) as cursor:
    for row in cursor:
        # set fields
        apn = row[0]
        if not (apn is None):
            row[1] = 0
        else:
            row[1] = 1
            cursor.updateRow(row)

print ("Finished calculating values")

# Execute Select
arcpy.Select_analysis(Join, FinalObs, '"Parcel_Master_DUPLICATE" = 1')

# Export to csv
finalfc = arcpy.da.FeatureClassToNumPyArray(FinalObs,['Parcel_Master_APN', 'Parcel_Master_PPNO', 'Parcel_Master_JURISDICTION'])
exportfc = pd.DataFrame(finalfc)

exportfc.to_csv(ObsoleteCSV)
print ("Exported csv table")



##New Parcels##

#Create a feature layer from featureclass
arcpy.MakeFeatureLayer_management (OldParcelLayer, ParcelJoin2)
print ("Made feature layer")

#Join between old parcel layer and new parcel layer
parcel_join = arcpy.AddJoin_management(ParcelJoin2, joinfield, ParcelLayer, joinfield)
print ("Finished Join")

#Write the joined features to a new featureclass
arcpy.CopyFeatures_management(ParcelJoin2, Join2)

#Find values that are new between last update and this update
field_names = ['Parcel_Master_APN_1', 'Parcel_Master_DUPLICATE']
with arcpy.da.UpdateCursor(Join2, field_names) as cursor:
    for row in cursor:
        # set fields
        apn = row[0]
        if (apn is None):
            row[1] = 1
        else:
            row[1] = 0
            cursor.updateRow(row)

print ("Finished calculating values")

#Execute Select
arcpy.Select_analysis(Join2, FinalNew, '"Parcel_Master_DUPLICATE" = 1')

#Export to csv
finalfc = arcpy.da.FeatureClassToNumPyArray(FinalNew,['Parcel_Master_APN', 'Parcel_Master_PPNO', 'Parcel_Master_JURISDICTION'])
exportfc = pd.DataFrame(finalfc)
exportfc.to_csv(NewCSV)
print ("Exported csv table")

# Delete temp feature classes
# if arcpy.Exists(Join):
#     arcpy.Delete_management(Join)
    
# if arcpy.Exists(Join2):
#     arcpy.Delete_management(Join2)
    
# if arcpy.Exists(FinalObs):
#     arcpy.Delete_management(FinalNew)
    
# if arcpy.Exists(ParcelJoin2):
#     arcpy.Delete_management(FinalNew)

### Check for NULL

In [ ]:
import arcpy, sys, datetime, os, traceback

#Set variables
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

#Check For Nulls
with arcpy.da.SearchCursor(ParcelLayer,"*") as cursor:
    for row in cursor:
        if None in row:
            print('I found a NULL value')
            break
            
###Note: Add print to tell which fields have a NULL value###
###Note: Add part to write over null values###

In [ ]:
###Note: Add print to tell which fields have a NULL value###
###Note: Add part to write over null values###
fieldObs = arcpy.ListFields(site_constraints_fc)  
fieldNames = []  
for field in fieldObs:  
    fieldNames.append(field.name)  
del fieldObs  
fieldCount = len(fieldNames)  

with arcpy.da.UpdateCursor(site_constraints_fc, fieldNames) as curU:  
    for row in curU:  
        rowU = row  
        for field in range(fieldCount):  
            if field == str:
                if rowU[field] == None or rowU[field] == "" or rowU[field].isspace == True:
                    rowU[field] = "" 
            else:
                if rowU[field] == None:
                    rowU[field] = 0 
        curU.updateRow(rowU)
    print("Complete.")

### Check for Duplicates

In [ ]:
import arcpy

print ("Checking for duplicates...")
def findDupes(ParcelLayer, checkField, updateField):
    with arcpy.da.SearchCursor(ParcelLayer, [checkField]) as rows:
        values = [r[0] for r in rows]

    with arcpy.da.UpdateCursor(ParcelLayer, [checkField, updateField]) as rows:
        for row in rows:
            if values.count(row[0]) > 1:
                row[1] = 1
                print ("Duplicate found")
            else:
                row[1] = 0
            rows.updateRow(row)

if __name__ == '__main__':
    ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
    checkField = "APN"
    updateField = "DUPLICATE"

    findDupes(ParcelLayer, checkField, updateField)
print ("Finished")

### Check for APN == PPNO

In [ ]:
import arcpy, sys, datetime, os, traceback

ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
expression = 'int(!APN!.replace("-",""))'

# Add field for calculation
arcpy.AddField_management(ParcelLayer, "APN_Calc", "TEXT", 20)
print ("Field Added")

# Remove the hyphens from APNs and create integer
arcpy.CalculateField_management(ParcelLayer, "APN_Calc", expression, "PYTHON3")
print ("Removed hyphens & created integer")

# Update duplicate field with 1 if the PPNO doesn't equal the APN
field_names = ['PPNO','APN_Calc', 'Duplicate']
with arcpy.da.UpdateCursor(ParcelLayer, field_names) as cursor:
    for row in cursor:
        # set fields
        ppno = row[0]
        apncheck = row[1]
        if not (ppno == int(apncheck)):
            row[2] = 1
            print ("Conflict detected")
        else:
            row[2] = 0
        cursor.updateRow(row)
print ("Comparison complete")

# Delete field
arcpy.DeleteField_management(ParcelLayer, "APN_Calc")
print ("Field Deleted")


### Check Topology

## Update Parcel Edit and Collection SDE

## Update SDE

In [ ]:
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
field_names = [field.name for field in arcpy.ListFields(inputfc)]
print (field_names)
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (field)

### Update Parcel Point

In [3]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\ParcelPoints"

# Change this to the path of your output FC
outfc = "F:\\GIS\\DB_CONNECT\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.ParcelPoints"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\DB_CONNECT\\Vector.sde\\sde.SDE.Parcels"

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@XY']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))


Imported arcpy, os, csv, and datatime modules.

ArcInfo

Disconnecting all users...

Deleted all records in: F:\GIS\DB_CONNECT\Vector.sde\sde.SDE.Parcels\sde.SDE.ParcelPoints

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into SDE feature class
Inserting record 18000 into SDE feature class
Inser

### Update Parcel Base

In [4]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcels_Base"

# Change this to the path of your output FC
outfc = "F:\\GIS\\DB_CONNECT\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcels_Base"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\DB_CONNECT\\Vector.sde\\sde.SDE.Parcels"

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeBase, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, os, csv, and datatime modules.

ArcInfo
Disconnecting Users...

Deleted all records in: F:\GIS\DB_CONNECT\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcels_Base

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into SDE feature class
Inserting record 18000 into SDE feature class
Inserting 

### Update Parcel Master

In [7]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

fieldnames = ['APN', 'PPNO', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME', 'STR_SUFFIX', 
               'APO_ADDRESS', 'PSTL_TOWN', 'PSTL_STATE', 'PSTL_ZIP5', 'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 
               'MAIL_ADD1', 'MAIL_ADD2', 'MAIL_CITY', 'MAIL_STATE', 'MAIL_ZIP5', 'JURISDICTION', 'COUNTY', 
               'OWNERSHIP_TYPE', 'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
               'REGIONAL_LANDUSE', 'UNITS', 'BEDROOMS', 'BATHROOMS', 'ALLOWABLE_COVERAGE_BAILEY_SQFT', 
               'IMPERVIOUS_SURFACE_SQFT', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 'WATERSHED_NUMBER', 'WATERSHED_NAME', 
               'PRIORITY_WATERSHED', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 'AS_LANDVALUE', 'AS_IMPROVALUE', 
               'AS_SUM', 'TAX_LANDVALUE', 'TAX_IMPROVALUE', 'TAX_SUM', 'TAX_YEAR', 'PLAN_ID', 'PLAN_NAME', 
               'ZONING_ID', 'ZONING_DESCRIPTION', 'TOLERANCE_ID', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'INDEX_1987', 
               'LOCAL_PLAN_HYPERLINK', 'DESIGN_GUIDELINES_HYPERLINK', 'LTINFO_LINK', 'INDEX_1987_HYPERLINK', 'TAZ', 
               'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

print ("Unregistering feature dataset as versioned...")
# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")
print ("Finished unregistering feature dataset as versioned.")
# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# # disconnect all users
# print("\nDisconnecting all users...")
# arcpy.DisconnectUser(sdeBase, "ALL")

print("\nRegistering feature dataset as versioned...")
# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")
print("\nFinished registering feature dataset as versioned.")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, os, csv, and datatime modules.

ArcInfo
Unregistering feature dataset as versioned...
Finished unregistering feature dataset as versioned.

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcel_Master

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into

### Update Parcel_Simplified

In [3]:
# Import system modules
import arcpy
import arcpy.management as DM
import arcpy.cartography as CA
import os, csv
from datetime import datetime

# start timer
startTimer = datetime.now()
print("Imported arcpy, arcpy.management as DM, arcpy.cartography as CA, os, csv, and datatime modules.\n")

# license used
print(arcpy.ProductInfo())

## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# in memory location
wk_memory = "in_memory" + "\\"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# in memory simplefied parcel master
simplifiedFeatures = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Simple"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcels_Simplified"
 
## ADD NEW FIELDS TO PARCEL_MASTER AND PARCEL_SIMPLIFIED!!!!
# set field map
fieldnames =  ['APN','PPNO','JURISDICTION','APO_ADDRESS','OWN_FULL','OWNERSHIP_TYPE','TRPA_LANDUSE_DESCRIPTION',
               'PLAN_ID', 'PLAN_NAME', 'ZONING_ID', 'ZONING_DESCRIPTION', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 
               'INDEX_1987', 'LOCAL_PLAN_HYPERLINK', 'DESIGN_GUIDELINES_HYPERLINK', 'INDEX_1987_HYPERLINK', 
               'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']

#--------------------------------------------------------------------------------------------------------# 
# create in memory simple parcels
CA.SimplifyPolygon(inputfc, simplifiedFeatures, "POINT_REMOVE", 1, "#", "#", "KEEP_COLLAPSED_POINTS")

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
print ("Unregistering feature dataset as versioned...")
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")
print ("Finished unregistering feature dataset as versioned.")

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(simplifiedFeatures, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

print("Registering feature dataset as versioned...")
# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")
print("Finished registering feature dataset as versioned.")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, arcpy.management as DM, arcpy.cartography as CA, os, csv, and datatime modules.

ArcInfo
Unregistering feature dataset as versioned...
Finished unregistering feature dataset as versioned.

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcels_Simplified

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16

In [4]:
import arcpy
table ="F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
field_names = [field.name for field in arcpy.ListFields(simplifiedFeatures)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

0 OBJECTID
1 Shape
2 APN
3 PPNO
4 HSE_NUMBR
5 UNIT_NUMBR
6 STR_DIR
7 STR_NAME
8 STR_SUFFIX
9 APO_ADDRESS
10 PSTL_TOWN
11 PSTL_STATE
12 PSTL_ZIP5
13 OWN_FIRST
14 OWN_LAST
15 OWN_FULL
16 MAIL_ADD1
17 MAIL_ADD2
18 MAIL_CITY
19 MAIL_STATE
20 MAIL_ZIP5
21 JURISDICTION
22 COUNTY
23 OWNERSHIP_TYPE
24 COUNTY_LANDUSE_CODE
25 COUNTY_LANDUSE_DESCRIPTION
26 TRPA_LANDUSE_DESCRIPTION
27 REGIONAL_LANDUSE
28 SOIL_1974
29 SOIL_2003
30 ALLOWABLE_COVERAGE_BAILEY_SQFT
31 IMPERVIOUS_SURFACE_SQFT
32 HRA_NAME
33 WATERSHED_NUMBER
34 WATERSHED_NAME
35 PRIORITY_WATERSHED
36 FIREPD
37 WITHIN_TRPA_BNDY
38 LITTORAL
39 AS_LANDVALUE
40 AS_IMPROVALUE
41 AS_SUM
42 TAX_LANDVALUE
43 TAX_IMPROVALUE
44 TAX_SUM
45 TAX_YEAR
46 PAS_ID
47 PAS_NAME
48 INDEX_1987
49 INDEX_1987_HYPERLINK
50 LOCAL_PLAN_HYPERLINK
51 ZONING
52 ZONING_DESCRIPTION
53 SINGLE_FAMILY_DENSITY
54 MULTI_FAMILY_DENSITY
55 TOURIST_ACCOMMODATION_DENSITY
56 BED_BREAKFAST_DENSITY
57 TIME_SHARE_DENSITY
58 COMMERCIAL_FLOOR_AREA_ALLOWED
59 SECONDARY_DWELLING_UNIT_

### UPDATE Parcel Tables

#### Update APO Table

In [ ]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Tables.gdb\\Parcel_APO"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Tabular.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_APO"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Tabular.sde

#IMPORTANT Fix PAS Id and PAS Name 
fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'OWN_FULL', 'APO_ADDRESS', 'OWNERSHIP_TYPE', 'TRPA_LANDUSE_DESCRIPTION',
              'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 'WATERSHED_NUMBER',
              'WATERSHED_NAME', 'PRIORITY_WATERSHED', 'PLAN_ID', 'PLAN_NAME', 'FIREPD', 'WITHIN_TRPA_BNDY',
              'LITTORAL', 'PARCEL_ACRES', 'PARCEL_SQFT', 'IPESScore']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeTabular, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeTabular, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

#### Update Address Table

In [ ]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Tables.gdb\\Parcel_Address"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Tabular.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Address"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Tabular.sde

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME',
              'STR_SUFFIX', 'APO_ADDRESS', 'PSTL_TOWN', 'PSTL_STATE', 'PSTL_ZIP5']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeTabular, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeTabular, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

#### Update Owner Table

In [ ]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Tables.gdb\\Parcel_Owner"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Tabular.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Owner"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Tabular.sde

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 'MAIL_ADD1',
              'MAIL_ADD2', 'MAIL_CITY', 'MAIL_STATE', 'MAIL_ZIP5']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeTabular, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeTabular, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

#### Update Value Table

In [ ]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Tables.gdb\\Parcel_Value"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Tabular.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Value"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Tabular.sde

fieldnames = ['APN', 'PPNO', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME', 'STR_SUFFIX', 'APO_ADDRESS', 'PSTL_TOWN',
              'PSTL_STATE', 'PSTL_ZIP5', 'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 'MAIL_ADD1', 'MAIL_ADD2', 'MAIL_CITY',
              'MAIL_STATE', 'MAIL_ZIP5', 'JURISDICTION', 'COUNTY', 'OWNERSHIP_TYPE', 'COUNTY_LANDUSE_CODE', 
              'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 'REGIONAL_LANDUSE', 'UNITS', 'BEDROOMS', 'BATHROOMS',
              'ALLOWABLE_COVERAGE_BAILEY_SQFT', 'IMPERVIOUS_SURFACE_SQFT', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME',
              'WATERSHED_NUMBER', 'WATERSHED_NAME', 'PRIORITY_WATERSHED', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 
              'AS_LANDVALUE', 'AS_IMPROVALUE', 'AS_SUM', 'TAX_LANDVALUE', 'TAX_IMPROVALUE', 'TAX_SUM', 'TAX_YEAR', 'PLAN_ID',
              'PLAN_NAME', 'ZONING_ID', 'ZONING_DESCRIPTION', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'INDEX_1987',
              'LOCAL_PLAN_HYPERLINK', 'DESIGN_GUIDELINES_HYPERLINK', 'INDEX_1987_HYPERLINK', 'PARCEL_ACRES', 'PARCEL_SQFT',
              'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeTabular, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeTabular, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

#### Update Master Table

In [ ]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Tables.gdb\\Parcel_Master"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Tabular.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Tabular.sde

fieldnames = ['APN', 'PPNO', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME', 'STR_SUFFIX', 
               'APO_ADDRESS', 'PSTL_TOWN', 'PSTL_STATE', 'PSTL_ZIP5', 'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 
               'MAIL_ADD1', 'MAIL_ADD2', 'MAIL_CITY', 'MAIL_STATE', 'MAIL_ZIP5', 'JURISDICTION', 'COUNTY', 
               'OWNERSHIP_TYPE', 'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
               'REGIONAL_LANDUSE', 'UNITS', 'BEDROOMS', 'BATHROOMS', 'ALLOWABLE_COVERAGE_BAILEY_SQFT', 
               'IMPERVIOUS_SURFACE_SQFT', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 'WATERSHED_NUMBER', 'WATERSHED_NAME', 
               'PRIORITY_WATERSHED', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 'AS_LANDVALUE', 'AS_IMPROVALUE', 
               'AS_SUM', 'TAX_LANDVALUE', 'TAX_IMPROVALUE', 'TAX_SUM', 'TAX_YEAR', 'PLAN_ID', 'PLAN_NAME', 
               'ZONING_ID', 'ZONING_DESCRIPTION', 'TOLERANCE_ID', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'INDEX_1987', 
               'LOCAL_PLAN_HYPERLINK', 'DESIGN_GUIDELINES_HYPERLINK', 'LTINFO_LINK', 'INDEX_1987_HYPERLINK', 'TAZ', 
               'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeTabular, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeTabular, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

## Issues Log

###  Spring 2019
* Check for Blank Land Use Values in El Dorado

* Merge Duplicates in El Dorado and Placer data

* Check for '0' in Placer PPNO

* Add # to valid Unit number and Null to ''

* Null to '' for all fields

* Script Select by Location of CSLT Values in Jurisdiction field, County field == EL, CC, PL, DG 

* Generate List of New APNs for Adelle

* Add Land Use Calculation NOT OWNERSHIP_TYPE = 'PRIVATE' AND TRPA_LANDUSE_DESCRIPTION = 'Vacant' 

* Fix Duplicates at the beginning of the script

* If Mail1 == Mail2 Then Mail2 is NULL

## References

### Accessing data using cursors
* http://pro.arcgis.com/en/pro-app/arcpy/get-started/data-access-using-cursors.htm

### Replacement for the field calculator
* https://gis.stackexchange.com/questions/177923/using-updatecursor-for-joined-field-in-arcpy/178168#178168
* https://community.esri.com/blogs/richard_fairhurst/2014/11/08/turbo-charging-data-manipulation-with-python-cursors-and-dictionaries

### Which way is faster?
* https://gis.stackexchange.com/questions/195197/which-is-the-faster-way-to-copy-data-to-another-feature-class-feature-class-to?utm_medium=organic&utm_source=google_rich_qa&utm_campaign=google_rich_qa

### Full Address Calculation
* https://community.esri.com/thread/42173
* https://bit.ly/2HsQL9x

### How to create the attribute join and update function:
* https://gis.stackexchange.com/questions/177923/using-updatecursor-for-joined-field-in-arcpy/178168?utm_medium=organic&utm_source=google_rich_qa&utm_campaign=google_rich_qa

### How to create the centroid feature layer:
* http://pro.arcgis.com/en/pro-app/arcpy/data-access/featureclasstonumpyarray.htm
* http://pro.arcgis.com/en/pro-app/arcpy/data-access/numpyarraytofeatureclass.htm

### How to use the Spatial Join tool:
* http://pro.arcgis.com/en/pro-app/tool-reference/analysis/spatial-join.htm

### How to use the Select by Location tool:
* http://desktop.arcgis.com/en/arcmap/10.3/tools/data-management-toolbox/select-layer-by-location.htm

### How to use the Calculate field tool:
* http://pro.arcgis.com/en/pro-app/tool-reference/data-management/calculate-field.htm

### How to Simplify Polygons
* http://pro.arcgis.com/en/pro-app/tool-reference/cartography/simplify-polygon.htm